# June validation recovery

This repeats the existing fixed June comparison after the Colab runtime expired.
The old private predictions and source aggregate are unavailable. An earlier run
found 355 duplicate-key groups outside its candidate population; this rerun
discloses that inspection. It is retrospective validation, not a blind test.

The model settings, May feature window and June outcome window are unchanged.
Every May-selected candidate is retained. The source audit must find zero
duplicate/count/coverage failures among those candidates. No deduplication occurs.
Run after restoring the verified October–May cache. Raw data and scores stay private.


In [ ]:
import os, sys, subprocess, importlib, importlib.metadata
from pathlib import Path
os.environ.setdefault('MPLCONFIGDIR', str(Path('work/private_cache/matplotlib').resolve()))
PINS = {'numpy':'2.2.6','pandas':'2.2.3','scipy':'1.16.3','scikit-learn':'1.7.2',
        'matplotlib':'3.10.7','duckdb':'1.4.1','huggingface_hub':'1.0.1'}
to_install=[]
for name,version in PINS.items():
    try: installed=importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: installed=None
    if installed!=version: to_install.append(f'{name}=={version}')
if to_install:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*to_install])
    raise RuntimeError('Setup finished. Restart the Colab session, then Run all. Do not continue below this cell yet.')
for name,expected in PINS.items():
    module=importlib.import_module('sklearn' if name=='scikit-learn' else name)
    if module.__version__!=expected:
        raise RuntimeError('Restart the Colab session, then Run all: '+name+' is still loaded at another version.')
print('Pinned runtime verified. No warehouse data has been accessed.')


In [ ]:
"""Transparent, prespecified comparators for the fixed Stage 2 outcome.

All references use training features only. Scores are priorities, not probabilities,
causes, or claims that editing a page will recover clicks.
"""
import hashlib
import numpy as np
import pandas as pd

BASELINE_FEATURES=['impressions_prior28','clicks_prior28','ctr_prior28','avg_position_prior28']
NEW_BASELINES=['Prior clicks','Prior impressions','Prior CTR',
               'Visible pooled-CTR shortfall','Visible pooled-CTR excess']


def validate_baseline_features(frame):
    x=frame[BASELINE_FEATURES].to_numpy(dtype=float)
    if not len(frame) or not np.isfinite(x).all():
        raise ValueError('Baseline input must be nonempty with finite prior-window features.')
    if (frame.impressions_prior28<=0).any() or (frame.clicks_prior28<=0).any():
        raise ValueError('Keep the fixed Stage 2 positive-count eligibility policy.')
    if (frame.clicks_prior28>frame.impressions_prior28).any():
        raise ValueError('Clicks exceed impressions; audit measurement before scoring.')
    if (frame.avg_position_prior28<=0).any():
        raise ValueError('Stage 2 requires positive observed positions.')
    if not np.allclose(x[:,2],100*x[:,1]/x[:,0]):
        raise ValueError('Prior CTR does not match prior clicks/impressions on the 0–100 scale.')


def fit_pooled_reference(train):
    validate_baseline_features(train)
    bucket=pd.cut(train.avg_position_prior28,[0,3,10,20,50,np.inf],include_lowest=True)
    totals=train.groupby(bucket,observed=False).agg(
        impressions=('impressions_prior28','sum'),clicks=('clicks_prior28','sum'),
        n_pages=('impressions_prior28','size'))
    rates=100*totals.clicks/totals.impressions.replace(0,np.nan)
    return {'rates':rates,
            'global_ctr':float(100*train.clicks_prior28.sum()/train.impressions_prior28.sum()),
            'visibility_threshold':float(train.impressions_prior28.median()),
            'training_bucket_support':[{'position_bucket':str(index),
                'n_pages':int(row.n_pages),'impressions':float(row.impressions),
                'clicks':float(row.clicks),
                'pooled_ctr':float(rates.loc[index]) if pd.notna(rates.loc[index]) else None}
                for index,row in totals.iterrows()]}


def score_transparent_baselines(reference,evaluation):
    validate_baseline_features(evaluation)
    bucket=pd.cut(evaluation.avg_position_prior28,[0,3,10,20,50,np.inf],include_lowest=True)
    expected=bucket.map(reference['rates']).astype(float)
    fallback=expected.isna()
    expected=expected.fillna(reference['global_ctr'])
    visible=(evaluation.impressions_prior28>=reference['visibility_threshold']).astype(int)
    scores={
        'Prior clicks':evaluation.clicks_prior28.to_numpy(dtype=float),
        'Prior impressions':evaluation.impressions_prior28.to_numpy(dtype=float),
        'Prior CTR':evaluation.ctr_prior28.to_numpy(dtype=float),
        'Visible pooled-CTR shortfall':(visible*(expected-evaluation.ctr_prior28).clip(lower=0)).to_numpy(dtype=float),
        'Visible pooled-CTR excess':(visible*(evaluation.ctr_prior28-expected).clip(lower=0)).to_numpy(dtype=float),
    }
    audit={'n_evaluation_pages':len(evaluation),
           'n_unseen_bucket_global_fallback':int(fallback.sum()),
           'training_global_ctr':reference['global_ctr'],
           'training_visibility_threshold':reference['visibility_threshold'],
           'training_bucket_support':reference['training_bucket_support']}
    return scores,audit


In [ ]:
"""Frozen retrospective time-ordered evaluation; no warehouse access on import."""
import hashlib
import json
from datetime import date, timedelta

import numpy as np
import pandas as pd

KEYS05=['client_hash_id','content_hash_id']
FEATURES05=['impressions_prior28','clicks_prior28','ctr_prior28','avg_position_prior28']
METHODS05=['Baseline rule','Logistic Regression','Random Forest','Prior clicks',
           'Prior impressions','Prior CTR','Visible pooled-CTR shortfall','Visible pooled-CTR excess']
TEST_ORIGINS05=[date(2026,m,1) for m in (2,3,4,5)]
EPISODES05=[date(2025,11,1),date(2025,12,1)]+[date(2026,m,1) for m in range(1,6)]
MONTHS05=['2025-10','2025-11','2025-12']+[f'2026-{m:02d}' for m in range(1,6)]
PROTOCOL05={
    'version':'fix05-v1', 'status':'frozen_retrospective_protocol_not_preregistration',
    'question':'Does a fixed learned ranking add retrieval value beyond simple rules on later periods?',
    'primary_learned_reference':'Logistic Regression','methods':METHODS05,
    'features':FEATURES05,'metadata_used':False,'ids_are_features':False,
    'origins':[str(t) for t in TEST_ORIGINS05],
    'development_origins':['2026-02-01','2026-03-01'],
    'later_origins':['2026-04-01','2026-05-01'],
    'later_origin_status':'Not scored by this repair workflow; prior user exposure is unknown. Not a blind-test claim.',
    'feature_window':'t-31 through t-4 inclusive, 28 days',
    'outcome_window':'t through t+27 inclusive, 28 days',
    'assumed_gap':'Three full date labels t-3,t-2,t-1; all training outcome dates must also be <=t-4.',
    'availability_limitation':'Retrospective final release; no historical ingestion vintages. Buffer is an assumption, not verified availability.',
    'date_convention':'Warehouse report_date labels, without timezone conversion; source timezone not independently verified.',
    'training':'Exactly the last three mature episode origins, pooled page-episodes; no shorter fallback history.',
    'training_guards':{'n_labelable_rows':500,'n_clients':2,'each_class':20},
    'primary_population':'As-of eligible pages of clients with a labelable row in the rolling training frame.',
    'eligibility':'28 explicit available prior daily rows; I>0, C>0, finite positive impression-weighted position. Sparse/missing days are not zero-filled.',
    'outcome_observability':'28 explicit available outcome rows with valid counts; otherwise unknown, retained in the as-of queue.',
    'label':'9*I_prior<=10*I_outcome<=11*I_prior and 20*C_outcome<=17*C_prior',
    'primary_budget':50,'secondary_budgets':[25,100,200],
    'primary_reporting':'Rank all eligible pages before outcome filtering. Report observed hits, unknowns, bounds; point precision only for fully observed queues.',
    'secondary_reporting':'Complete-case metrics, support >=5/20 without refitting, anonymous client contributions.',
    'tie_policy':'Descending score, stable canonical client/content key order; conditional tie ranges also reported.',
    'hash_holdout':'sha256(flyrank-fix05-v1|client_id) mod 4 == 0; excluded from all hash-arm training; descriptive.',
    'models':{'LR':{'C':1.0,'max_iter':1000,'seed':42},
              'RF':{'n_estimators':300,'seed':42,'n_jobs':-1,'other_parameters':'sklearn 1.7.2 defaults'},
              'preprocessing':'Independent StandardScaler per model, four numeric features only; no tuning or winner selection.'},
    'rules':'Training-only position bands (0,3,10,20,50,infinity); medians or pooled CTR; visibility at training median I. High prior counts/CTR rank first.',
    'unseen_band_fallback':'Global training median CTR for original rule; global pooled training CTR for pooled rules.',
    'source_months':MONTHS05,'final_month':'June 2026 is excluded, including the sample table.',
    'inference_limits':'Repeated pages and clients; no iid page confidence intervals. Few periods/clients may preclude a strong claim.',
    'changes_from_stage2':'Numeric-only features, lagged schedule, rolling fitting, different client split and as-of queues. Changes cannot be attributed solely to time splitting.'}


def protocol_hash05():
    return hashlib.sha256(json.dumps(PROTOCOL05,sort_keys=True).encode()).hexdigest()


def windows05():
    rows=[]
    for t in EPISODES05:
        for role,start,end in [('feature',t-timedelta(days=31),t-timedelta(days=4)),
                               ('outcome',t,t+timedelta(days=27))]:
            rows.append({'origin':str(t),'role':role,'start_date':start,'end_date':end})
    return pd.DataFrame(rows)


def train_origins05(t):
    return [s for s in EPISODES05 if s<t and s+timedelta(days=27)<=t-timedelta(days=4)][-3:]


def heldout05(client):
    return int(hashlib.sha256(('flyrank-fix05-v1|'+str(client)).encode()).hexdigest(),16)%4==0


def build_episode05(aggregate,origin):
    a=aggregate.copy()
    if a.duplicated(['role']+KEYS05).any() or a[KEYS05].isna().any().any():
        raise ValueError('Invalid aggregate grain or missing page keys.')
    if (a.raw_rows!=a.raw_days).any():
        raise ValueError('Duplicate daily keys; stop before modelling.')
    if (a.bad_counts>0).any():
        raise ValueError('Invalid available count data; stop before modelling.')
    for column in ['impressions','clicks']:
        values=a[column].dropna().to_numpy(dtype=float)
        if ((values<0)|(values>10**14)|(values!=np.floor(values))|~np.isfinite(values)).any():
            raise ValueError('Count range/integer guard failed.')
    f=a.loc[a.role.eq('feature')].copy()
    o=a.loc[a.role.eq('outcome'),KEYS05+['available_days','impressions','clicks']].copy()
    o=o.rename(columns={c:c+'_outcome' for c in ['available_days','impressions','clicks']})
    position=f.sum_position/f.impressions.replace(0,np.nan)
    full=f.available_days.eq(28)
    positive=full&f.impressions.gt(0)&f.clicks.gt(0)
    eligible=positive&f.bad_position.eq(0)&np.isfinite(position)&position.gt(0)
    flow=[]
    for name,mask in [('all_prior_pages',pd.Series(True,index=f.index)),
                      ('complete_prior_rows',full),('positive_prior_counts',positive),
                      ('as_of_eligible',eligible)]:
        flow.append({'stage':name,'n_pages':int(mask.sum()),'n_clients':int(f.loc[mask,'client_hash_id'].nunique())})
    f=f.loc[eligible].copy()
    f['avg_position_prior28']=position.loc[eligible]
    f=f.rename(columns={'impressions':'impressions_prior28','clicks':'clicks_prior28'})
    f['ctr_prior28']=100*f.clicks_prior28/f.impressions_prior28
    d=f[KEYS05+FEATURES05].merge(o,on=KEYS05,how='left',validate='one_to_one')
    observed=d.available_days_outcome.eq(28)
    for col in ['impressions_prior28','clicks_prior28','impressions_outcome','clicks_outcome']:
        d[col]=d[col].astype('Int64')
    label=((9*d.impressions_prior28<=10*d.impressions_outcome)&
           (10*d.impressions_outcome<=11*d.impressions_prior28)&
           (20*d.clicks_outcome<=17*d.clicks_prior28)).astype('Int64')
    d['decoupling_signature']=label.where(observed,pd.NA)
    d['origin']=str(origin)
    d=d.sort_values(KEYS05,kind='stable').reset_index(drop=True)
    return d,{'origin':str(origin),'flow':flow,'n_labelable':int(observed.sum()),
              'n_unknown':int((~observed).sum()),'n_positive':int(d.decoupling_signature.sum())}


def score_models05(train,evaluation):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    import warnings
    from sklearn.exceptions import ConvergenceWarning
    from threadpoolctl import threadpool_limits
    # No outcome or identifier column is passed to either model.
    scores={}
    bucket=pd.cut(train.avg_position_prior28,[0,3,10,20,50,np.inf],include_lowest=True)
    median=train.groupby(bucket,observed=False).ctr_prior28.median()
    e_bucket=pd.cut(evaluation.avg_position_prior28,[0,3,10,20,50,np.inf],include_lowest=True)
    expected=e_bucket.map(median).astype(float).fillna(float(train.ctr_prior28.median()))
    scores['Baseline rule']=((evaluation.impressions_prior28>=train.impressions_prior28.median())*
                            (expected-evaluation.ctr_prior28).clip(lower=0)).to_numpy(dtype=float)
    for name,clf in [('Logistic Regression',LogisticRegression(C=1.0,max_iter=1000,random_state=42)),
                     ('Random Forest',RandomForestClassifier(n_estimators=300,random_state=42,n_jobs=-1))]:
        model=make_pipeline(StandardScaler(),clf)
        with warnings.catch_warnings(),threadpool_limits(limits=1):
            warnings.simplefilter('error',ConvergenceWarning)
            model.fit(train[FEATURES05].to_numpy(dtype=float),train.decoupling_signature.to_numpy(dtype=int))
            scores[name]=model.predict_proba(evaluation[FEATURES05].to_numpy(dtype=float))[:,1]
    rules,audit=score_transparent_baselines(fit_pooled_reference(train),evaluation)
    return {**scores,**rules},audit


def metrics05(frame,scores,ks=(25,50,100,200)):
    from sklearn.metrics import roc_auc_score,average_precision_score
    s=np.asarray(scores,dtype=float)
    if len(s)!=len(frame) or not np.isfinite(s).all():
        raise ValueError('Non-finite or unaligned predictions.')
    known=frame.decoupling_signature.notna().to_numpy()
    y=frame.decoupling_signature.fillna(0).to_numpy(dtype=int)
    order=np.argsort(-s,kind='stable')
    complete_order=np.flatnonzero(known)[np.argsort(-s[known],kind='stable')]
    positives=int(y[known].sum()); n=int(known.sum())
    base=float(positives/n) if n else None
    out={'n_as_of':len(frame),'n_observed':n,'n_unknown':len(frame)-n,
         'observed_positive_count':positives,'complete_case_prevalence':base,
         'roc_auc_complete_case':float(roc_auc_score(y[known],s[known])) if n and np.unique(y[known]).size==2 else None,
         'average_precision_complete_case':float(average_precision_score(y[known],s[known])) if positives else None,
         'queues':[],'complete_case_rank_metrics':[]}
    for k in ks:
        if len(s)<k:
            queue={'budget':k,'reason':'insufficient_as_of_pages','reviewed':None}
        else:
            ix=order[:k]; hits=int(y[ix].sum()); unknown=int((~known[ix]).sum())
            cutoff=s[ix[-1]]; above=s>cutoff; tied=s==cutoff
            slots=k-int(above.sum()); tie_n=int(tied.sum()); tie_hits=int(y[tied].sum())
            tie_unknown=int((tied&~known).sum()); above_hits=int(y[above].sum())
            above_unknown=int((above&~known).sum())
            queue={'budget':k,'reason':None,'reviewed':k,'observed_hits':hits,'unknown_outcomes':unknown,
                   'label_coverage':(k-unknown)/k,'precision_if_complete':hits/k if unknown==0 else None,
                   'precision_lower_bound':hits/k,'precision_upper_bound':(hits+unknown)/k,
                   'cutoff_tie_count':tie_n,
                   'tie_and_missing_lower_bound':(above_hits+max(0,slots-(tie_n-tie_hits)))/k,
                   'tie_and_missing_upper_bound':(above_hits+above_unknown+min(slots,tie_hits+tie_unknown))/k}
        out['queues'].append(queue)
        if n<k:
            cc={'budget':k,'precision':None,'reason':'insufficient_observed_pages'}
        else:
            hits=int(y[complete_order[:k]].sum())
            cc={'budget':k,'hits':hits,'precision':hits/k,'lift':hits/k/base if base else None,'reason':None}
        out['complete_case_rank_metrics'].append(cc)
    return out


def cohort_summary05(frame):
    n=len(frame); known=frame.decoupling_signature.notna()
    sizes=frame.groupby('client_hash_id').size().sort_values(ascending=False)
    return {'n_page_episodes':n,'n_clients':len(sizes),'n_labelable':int(known.sum()),
            'n_positive':int(frame.decoupling_signature.sum()),
            'n_clients_with_at_least_50_pages':int((sizes>=50).sum()),
            'largest_two_client_fraction':float(sizes.iloc[:2].sum()/n) if n else None}


def run_chronological05(episodes,cohort_audits,score_sink=None,progress=None):
    results=[]
    # Names are export-only aliases, never predictors or split inputs.
    roster=sorted({str(x) for frame in episodes.values() for x in frame.client_hash_id})
    aliases={client:f'client_{i+1:03d}' for i,client in enumerate(roster)}
    for t in TEST_ORIGINS05:
        origins=train_origins05(t)
        fold={'origin':str(t),'cutoff':str(t-timedelta(days=4)),
              'training_origins':[str(s) for s in origins],'arms':[]}
        results.append(fold)
        if len(origins)!=3 or any(s not in episodes or not episodes[s].decoupling_signature.notna().any() for s in origins):
            fold['status']='insufficient_history';continue
        train=pd.concat([episodes[s] for s in origins],ignore_index=True)
        train=train.loc[train.decoupling_signature.notna()].copy()
        future=episodes[t].copy()
        for arm in ['known_clients','naturally_unseen_clients','hash_heldout_clients']:
            tr=train.copy()
            if arm=='hash_heldout_clients':
                tr=tr.loc[~tr.client_hash_id.map(heldout05)]
                test=future.loc[future.client_hash_id.map(heldout05)].copy()
            else:
                seen=future.client_hash_id.isin(set(tr.client_hash_id))
                test=future.loc[seen if arm=='known_clients' else ~seen].copy()
            test=test.sort_values(KEYS05,kind='stable').reset_index(drop=True)
            record={'arm':arm,'training':cohort_summary05(tr),'evaluation':cohort_summary05(test),'metrics':[]}
            fold['arms'].append(record)
            if set(tr.origin)!={str(s) for s in origins}:
                record['status']='insufficient_training_history';continue
            counts=tr.decoupling_signature.value_counts()
            if len(tr)<500 or tr.client_hash_id.nunique()<2 or min(counts.get(0,0),counts.get(1,0))<20:
                record['status']='insufficient_training_support';continue
            if not len(test):
                record['status']='no_eligible_test_pages';continue
            if arm=='hash_heldout_clients' and set(tr.client_hash_id)&set(test.client_hash_id):
                raise AssertionError('Hash holdout client leakage.')
            if progress is not None:
                progress(f'Fitting {t} {arm}')
            scores,reference_audit=score_models05(tr,test)
            if score_sink is not None:
                score_sink(str(t),arm,test,scores)
            record['status']='scored';record['reference_audit']=reference_audit
            record['prediction_sha256']={name:hashlib.sha256(np.asarray(s,dtype=float).tobytes()).hexdigest()
                                          for name,s in scores.items()}
            record['cohort_sha256']=hashlib.sha256(pd.util.hash_pandas_object(
                test[KEYS05+FEATURES05+['decoupling_signature']],index=False).values.tobytes()).hexdigest()
            for minimum in (1,5,20):
                mask=test.clicks_prior28.ge(minimum).to_numpy(dtype=bool)
                for name,s in scores.items():
                    record['metrics'].append({'method':name,'minimum_prior_clicks':minimum,
                                               'support_population':cohort_summary05(test.loc[mask]),
                                               **metrics05(test.loc[mask],s[mask])})
            details=[]
            global_orders={name:np.argsort(-s,kind='stable')[:50] if len(test)>=50 else np.array([],dtype=int)
                           for name,s in scores.items()}
            for client,part in test.groupby('client_hash_id',sort=True):
                for name,order in global_orders.items():
                    selected=test.iloc[order]; selected=selected.loc[selected.client_hash_id.eq(client)]
                    details.append({'anonymous_client':aliases[str(client)],'method':name,
                        **cohort_summary05(part),'global_queue_pages':len(selected),
                        'global_queue_observed_hits':int(selected.decoupling_signature.sum()),
                        'global_queue_unknown':int(selected.decoupling_signature.isna().sum())})
            record['anonymous_client_details']=details
        fold['status']='completed_with_explicit_arm_statuses'
    return {'protocol':PROTOCOL05,'protocol_sha256':protocol_hash05(),
            'cohort_audits':cohort_audits,'folds':results,
            'limitations':['Retrospective final release, assumed availability buffer.',
                           'Outcome missingness is not assumed random; main queue bounds are descriptive.',
                           'Repeated page-episodes and few clients/periods; no iid uncertainty claims.',
                           'No causal attribution or evidence that editing pages improves outcomes.']}


In [ ]:
"""Execution-only repair: bounded, resumable monthly aggregation for frozen Fix05."""
from datetime import date
from pathlib import Path
import gc,hashlib,json,os,shutil,uuid


def file_hash05m(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(4*1024*1024),b''):digest.update(block)
    return digest.hexdigest()


def sql_literal05m(value):
    return "'"+str(value).replace("'","''")+"'"


def parquet05m(paths):
    return 'read_parquet(['+','.join(sql_literal05m(Path(p).as_posix() if isinstance(p,Path) else p) for p in paths)+'],hive_partitioning=true)'


def month_bounds05m(month):
    year,number=map(int,month.split('-'))
    start=date(year,number,1)
    end=date(year+1,1,1) if number==12 else date(year,number+1,1)
    return start,end


def memory_settings05m(con,root):
    """Leave space for Python and allocations outside DuckDB's memory manager."""
    gc.collect(); candidates=[]
    try:
        info=dict(line.split(':',1) for line in Path('/proc/meminfo').read_text().splitlines())
        candidates.append(int(info['MemAvailable'].strip().split()[0])*1024)
    except (OSError,KeyError,ValueError):pass
    for limit_path,used_path in [('/sys/fs/cgroup/memory.max','/sys/fs/cgroup/memory.current'),
                                 ('/sys/fs/cgroup/memory/memory.limit_in_bytes','/sys/fs/cgroup/memory/memory.usage_in_bytes')]:
        try:
            limit=Path(limit_path).read_text().strip()
            if limit!='max':candidates.append(max(0,int(limit)-int(Path(used_path).read_text().strip())))
        except (OSError,ValueError):pass
    available=min(candidates) if candidates else None
    if available is not None and available<1024**3:
        raise RuntimeError('Less than 1 GiB available RAM. Save the notebook, restart the session, then use the memory-safe standalone notebook.')
    mib=min(4096,max(512,int(.4*available/1024**2))) if available else 1536
    root=Path(root);root.mkdir(parents=True,exist_ok=True)
    if shutil.disk_usage(root).free<2*1024**3:
        raise RuntimeError('Less than 2 GiB free disk for checkpoints/spill. Free space outside the verified recovery cache before retrying.')
    con.execute(f"SET memory_limit='{mib}MiB'")
    con.execute('SET threads=1')
    con.execute('SET preserve_insertion_order=false')
    con.execute('SET temp_directory='+sql_literal05m((root/'spill').resolve().as_posix()))
    return {'duckdb_memory_limit_mib':mib,'threads':1,'preserve_insertion_order':False,
            'available_memory_bytes_at_configuration':available}


def month_sql05m(source,month=None):
    # Every offset is relative to its own 28-day window. Exact bit sets replace
    # two large COUNT(DISTINCT ...) states per page/window; absence stays absent.
    scan=parquet05m(source)
    checked_date='d.report_date'
    if month is not None:
        start,end=month_bounds05m(month)
        # Validate in the expression used by the join, so out-of-window corrupt
        # dates cannot disappear before the audit. No separate remote date scan.
        scan=f"""(SELECT *,CASE WHEN report_date IS NOT NULL AND
          report_date>=DATE '{start}' AND report_date<DATE '{end}' THEN report_date
          ELSE error('Source partition contains null/out-of-month dates; stop before combining months.')
          END AS _checked_report_date FROM {scan})"""
        checked_date='d._checked_report_date'
    return f"""WITH grouped AS (
      SELECT w.origin,w.role,d.client_hash_id,d.content_hash_id,COUNT(*) AS raw_rows,
        bit_or(1::BIGINT << CAST(date_diff('day',w.start_date,{checked_date}) AS INTEGER)) AS row_mask,
        bit_or(CASE WHEN gsc_data_available IS TRUE THEN
          1::BIGINT << CAST(date_diff('day',w.start_date,{checked_date}) AS INTEGER)
          ELSE 0::BIGINT END) AS available_mask,
        COUNT(*) FILTER(WHERE gsc_data_available IS TRUE AND
          (gsc_impressions IS NULL OR gsc_clicks IS NULL OR
           NOT isfinite(gsc_impressions) OR NOT isfinite(gsc_clicks) OR
           gsc_impressions<0 OR gsc_clicks<0 OR gsc_clicks>gsc_impressions OR
           gsc_impressions!=floor(gsc_impressions) OR gsc_clicks!=floor(gsc_clicks))) AS bad_counts,
        COUNT(*) FILTER(WHERE gsc_data_available IS TRUE AND gsc_impressions>0 AND
          (gsc_sum_position IS NULL OR NOT isfinite(gsc_sum_position) OR gsc_sum_position<0)) AS bad_position,
        CAST(SUM(gsc_impressions) FILTER(WHERE gsc_data_available IS TRUE) AS BIGINT) AS impressions,
        CAST(SUM(gsc_clicks) FILTER(WHERE gsc_data_available IS TRUE) AS BIGINT) AS clicks,
        SUM(gsc_sum_position) FILTER(WHERE gsc_data_available IS TRUE) AS sum_position
      FROM {scan} d
      JOIN _fix05_windows w ON {checked_date} BETWEEN w.start_date AND w.end_date
      GROUP BY 1,2,3,4)
      SELECT *,CAST(bit_count(row_mask) AS BIGINT) AS raw_days,
        CAST(bit_count(available_mask) AS BIGINT) AS available_days FROM grouped"""


def merge_sql05m(paths,origin,role):
    return f"""WITH grouped AS (
      SELECT origin,role,client_hash_id,content_hash_id,
        CAST(SUM(raw_rows) AS BIGINT) AS raw_rows,
        bit_or(row_mask) AS row_mask,bit_or(available_mask) AS available_mask,
        CAST(SUM(raw_days) AS BIGINT) AS partial_day_total,
        CAST(SUM(bad_counts) AS BIGINT) AS bad_counts,
        CAST(SUM(bad_position) AS BIGINT) AS bad_position,
        CAST(SUM(impressions) AS BIGINT) AS impressions,
        CAST(SUM(clicks) AS BIGINT) AS clicks,SUM(sum_position) AS sum_position
      FROM {parquet05m(paths)}
      WHERE origin={sql_literal05m(origin)} AND role={sql_literal05m(role)}
      GROUP BY 1,2,3,4)
      SELECT *,CAST(bit_count(row_mask) AS BIGINT) AS raw_days,
        CAST(bit_count(available_mask) AS BIGINT) AS available_days FROM grouped"""


def validate_aggregate05m(con,path,merged=False):
    extra=' OR partial_day_total!=raw_days' if merged else ''
    n,bad=con.sql(f"""SELECT COUNT(*),COUNT(*) FILTER(WHERE
      client_hash_id IS NULL OR content_hash_id IS NULL OR raw_rows!=raw_days OR
      raw_days!=bit_count(row_mask) OR available_days!=bit_count(available_mask) OR
      raw_days>28 OR row_mask>=268435456 OR available_days>raw_days OR
      (row_mask | available_mask)!=row_mask OR bad_counts>0{extra})
      FROM {parquet05m([path])}""").fetchone()
    if bad:raise RuntimeError('Aggregate coverage/grain/count audit failed. Invalid groups: '+str(bad))
    return int(n)


def checked_checkpoint05m(path,identity):
    path=Path(path);receipt=path.with_suffix('.json')
    if not receipt.exists():return None
    metadata=json.loads(receipt.read_text())
    if (not path.exists() or any(metadata.get(k)!=v for k,v in identity.items()) or
        metadata.get('sha256')!=file_hash05m(path)):
        raise RuntimeError('Recovery checkpoint identity/checksum mismatch; preserve files and investigate.')
    return metadata


def write_checkpoint05m(con,query,path,identity,merged=False,validate=True):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    partial=path.with_name(path.stem+'.partial-'+uuid.uuid4().hex+'.parquet')
    con.execute('COPY ('+query+') TO '+sql_literal05m(partial.resolve().as_posix())+' (FORMAT PARQUET, COMPRESSION ZSTD)')
    rows=validate_aggregate05m(con,partial,merged) if validate else None
    metadata={**identity,'sha256':file_hash05m(partial),'n_aggregate_rows':rows}
    if path.exists():
        # Unverified interrupted outputs are preserved, never consumed as checkpoints.
        path.replace(path.with_name(path.stem+'.unverified-'+uuid.uuid4().hex+'.parquet'))
    partial.replace(path)
    receipt=path.with_suffix('.json');pending=receipt.with_suffix('.json.partial')
    pending.write_text(json.dumps(metadata,indent=2)+'\n',encoding='utf-8');pending.replace(receipt)
    return metadata


def build_cache05m(con,windows,months,root,protocol_sha,implementation_sha,source_paths,current_revision,progress=print):
    """Bound remote aggregation by month; local merge by window; durable checksums."""
    root=Path(root);root.mkdir(parents=True,exist_ok=True)
    identity={'protocol_sha256':protocol_sha,'implementation_sha256':implementation_sha,'source_months':list(months)}
    plan_path=root/'plan.json'
    if plan_path.exists():
        plan=json.loads(plan_path.read_text())
        if any(plan.get(k)!=v for k,v in identity.items()):
            raise RuntimeError('Recovery plan differs from verified checkpoints. Do not mix implementations or protocols.')
    else:
        plan={**identity,'dataset_revision':current_revision()}
        plan_path.write_text(json.dumps(plan,indent=2)+'\n',encoding='utf-8')
    identity={**identity,'dataset_revision':plan['dataset_revision']}
    final=root/'windows.parquet'
    final_identity={**identity,'kind':'combined_verified_windows'}
    ready=checked_checkpoint05m(final,final_identity)
    if ready:
        progress('Reusing verified recovery cache; no warehouse scan.')
        return final,ready
    month_paths=[];month_receipts=[]
    for number,month in enumerate(months,1):
        path=root/'months'/f'{month}.parquet';part_identity={**identity,'kind':'month','source_month':month}
        receipt=checked_checkpoint05m(path,part_identity)
        if receipt is None:
            if current_revision()!=plan['dataset_revision']:
                raise RuntimeError('Warehouse revision differs from existing checkpoints; mixed revisions are forbidden.')
            start,end=month_bounds05m(month);sources=source_paths(month)
            selected=windows.loc[(windows.start_date<end)&(windows.end_date>=start)]
            if not len(selected):raise ValueError('Frozen month has no intersecting windows.')
            con.register('_fix05_windows',selected)
            progress(f'Month {number}/{len(months)}: aggregating {month}; completed months are checkpointed.')
            # Stage the file first. A revision change must not bless its receipt.
            stage=root/'pending'/f'{month}.parquet';stage.parent.mkdir(parents=True,exist_ok=True)
            staged=write_checkpoint05m(con,month_sql05m(sources,month),stage,part_identity)
            if current_revision()!=plan['dataset_revision']:
                raise RuntimeError('Warehouse revision changed during this month; pending files will not be reused.')
            path.parent.mkdir(parents=True,exist_ok=True)
            if path.exists():path.replace(path.with_name(path.stem+'.unverified-'+uuid.uuid4().hex+'.parquet'))
            stage.replace(path);stage.with_suffix('.json').replace(path.with_suffix('.json'))
            receipt=staged
        else:progress(f'Month {number}/{len(months)}: reusing verified {month} checkpoint.')
        month_paths.append(path);month_receipts.append(receipt)
    window_paths=[]
    for _,window in windows.iterrows():
        origin,role=window.origin,window.role
        path=root/'windows'/f'{origin}-{role}.parquet'
        part_identity={**identity,'kind':'window','origin':str(origin),'role':str(role)}
        if checked_checkpoint05m(path,part_identity) is None:
            selected_paths=[p for p,month in zip(month_paths,months)
                if month_bounds05m(month)[0]<=window.end_date and month_bounds05m(month)[1]>window.start_date]
            progress('Merging private local aggregates: '+str(origin)+' '+str(role))
            write_checkpoint05m(con,merge_sql05m(selected_paths,origin,role),path,part_identity,merged=True)
        window_paths.append(path)
    # No global GROUP BY: concatenate verified window files as a streaming COPY.
    metadata=write_checkpoint05m(con,'SELECT * FROM '+parquet05m(window_paths),final,
        {**final_identity,'month_checkpoints':[{'source_month':r['source_month'],'sha256':r['sha256']}
                                              for r in month_receipts]},validate=False)
    return final,metadata


In [ ]:
"""Descriptive conditional count-allocation audit; never a forecast feature."""
import hashlib,json
import numpy as np
import pandas as pd

MAX_TOTAL_CLICKS06=200
COUNT_PROTOCOL06={
    'version':'fix06-v1','status':'exploratory_reference_audit_after_Fix05_review',
    'purpose':'Describe raw event counts against a simple stationary click-allocation reference.',
    'source':'Existing Fix05 page-episode aggregates and unchanged cohorts; no new warehouse query or model fit.',
    'population':'Complete-case members of each fixed Fix05 arm, restricted to 1<=prior+future clicks<=200.',
    'cap_reason':'Predeclared computational scope for the low-total-count diagnostic; total clicks are conditioned on by the reference.',
    'null':'Independent Bernoulli clicks per impression with one common page CTR in the two windows.',
    'conditioning':'Both impression totals, the combined click total, and prior clicks>0.',
    'reference':'Future clicks X ~ Hypergeometric(M=I0+I1,n=C0+C1,N=I1), truncated to X<=C0+C1-1.',
    'signature_probability':'If stable impressions, F(floor(17*(C0+C1)/37))/F(C0+C1-1); otherwise 0.',
    'computation':'Finite-support hypergeometric weights from log-ratio recurrence; batches of at most 4096 rows.',
    'reporting':'Observed signatures beside sum of conditional probabilities on exactly the same modeled subset.',
    'prior_click_bands':'Observed counts only. No reference expectation is reported by observed prior-click band.',
    'not_forecast':'Reference uses future impressions and clicks. It is outcome analysis, never a predictor or ranking baseline.',
    'limitations':['The common-CTR independent-click model may fail under temporal dependence, changing query mix, seasonality or heterogeneous exposure.',
                   'Complete-case retention must be ignorable under this conditional reference; outcome-dependent observation can invalidate it.',
                   'Other eligibility/arm membership is treated as fixed under the reference.',
                   'Total-click restriction uses future counts and does not redefine the main prediction population.',
                   'No false-positive fraction, equivalence, calibrated hypothesis test, confidence interval or causal attribution follows.',
                   'Arms overlap and monthly pages/clients repeat; do not pool them as independent evidence.']}


def conditional_signature_probability06(i0,c0,i1,c1):
    arrays=[np.asarray(x,dtype=float) for x in [i0,c0,i1,c1]]
    if any(x.ndim!=1 for x in arrays) or len({len(x) for x in arrays})!=1:
        raise ValueError('Count arrays must be aligned one-dimensional arrays.')
    if any(not np.isfinite(x).all() or ((x<0)|(x>10**14)|(x!=np.floor(x))).any() for x in arrays):
        raise ValueError('Finite nonnegative integer counts within the audited range are required.')
    i0,c0,i1,c1=[x.astype('int64') for x in arrays]
    if ((i0<=0)|(c0<=0)|(c0>i0)|(c1>i1)).any():
        raise ValueError('Preserve positive-prior-click eligibility and clicks<=impressions.')
    total=c0+c1
    if (total>MAX_TOTAL_CLICKS06).any():raise ValueError('Apply the declared total-click scope before this reference.')
    stable=(9*i0<=10*i1)&(10*i1<=11*i0)
    q=np.zeros(len(total),dtype=float)
    if stable.any():
        selected=np.flatnonzero(stable)
        for start in range(0,len(selected),4096):
            ix=selected[start:start+4096];t=total[ix];a=i0[ix];b=i1[ix]
            lower=np.maximum(0,t-a);upper=np.minimum(b,t-1)
            if (lower>upper).any():raise ValueError('Prior-positive conditional support is empty.')
            width=int((upper-lower+1).max())
            log_weights=np.full((len(ix),width),-np.inf);log_weights[:,0]=0.
            for j in range(1,width):
                valid=lower+j<=upper;x=lower[valid]+j-1
                # w(x+1)/w(x) for C(b,x)*C(a,t-x); the unknown normalizer cancels.
                log_weights[valid,j]=(log_weights[valid,j-1]+np.log(b[valid]-x)
                    +np.log(t[valid]-x)-np.log(x+1)-np.log(a[valid]-t[valid]+x+1))
            weights=np.exp(log_weights-log_weights.max(axis=1,keepdims=True))
            numerator=(weights*((lower[:,None]+np.arange(width))<=((17*t)//37)[:,None])).sum(axis=1)
            values=numerator/weights.sum(axis=1)
            if not np.isfinite(values).all() or ((values<0)|(values>1+1e-12)).any():
                raise ValueError('Conditional probability calculation failed.')
            q[ix]=np.minimum(values,1.)
    return q


def observed_counts06(frame):
    known=frame.decoupling_signature.notna()
    positive=int(frame.loc[known,'decoupling_signature'].sum())
    return {'n_pages':len(frame),'n_observed':int(known.sum()),'n_unknown':int((~known).sum()),
            'observed_positive_count':positive,'observed_event_rate':positive/int(known.sum()) if known.any() else None}


def summarize_count_noise06(frame):
    observed=frame.loc[frame.decoupling_signature.notna()].copy()
    fields=['impressions_prior28','clicks_prior28','impressions_outcome','clicks_outcome']
    counts=observed[fields].to_numpy(dtype=float)
    if not np.isfinite(counts).all() or ((counts<0)|(counts>10**14)|(counts!=np.floor(counts))).any():
        raise ValueError('Invalid complete-case counts.')
    i0,c0,i1,c1=counts.astype('int64').T
    if ((i0<=0)|(c0<=0)|(c0>i0)|(c1>i1)).any():raise ValueError('Invalid click/impression counts.')
    actual=((9*i0<=10*i1)&(10*i1<=11*i0)&(20*c1<=17*c0)).astype(int)
    if not np.array_equal(actual,observed.decoupling_signature.to_numpy(dtype=int)):
        raise ValueError('Observed labels differ from the frozen integer rule.')
    in_scope=(c0+c1)<=MAX_TOTAL_CLICKS06
    modeled=observed.loc[in_scope].copy()
    q=conditional_signature_probability06(i0[in_scope],c0[in_scope],i1[in_scope],c1[in_scope])
    count=int(modeled.decoupling_signature.sum());expected=float(q.sum());n=len(modeled)
    bands=[]
    for name,low,high in [('1-4',1,5),('5-19',5,20),('20+',20,np.inf)]:
        part=frame.loc[frame.clicks_prior28.ge(low)&frame.clicks_prior28.lt(high)]
        bands.append({'prior_click_band':name,**observed_counts06(part)})
    return {'full_arm_observation':observed_counts06(frame),
        'reference_subset':{'n_modeled_pages':n,'n_observed_pages_excluded_total_clicks_gt_200':int((~in_scope).sum()),
            'fraction_of_observed_pages_modeled':n/len(observed) if len(observed) else None,
            'n_stable_impression_pages':int(((9*i0[in_scope]<=10*i1[in_scope])&(10*i1[in_scope]<=11*i0[in_scope])).sum()),
            'observed_signature_count':count,'stationary_reference_expected_signature_count':expected,
            'observed_minus_reference_expected_count_descriptive':count-expected,
            'observed_signature_rate':count/n if n else None,
            'stationary_reference_expected_signature_rate':expected/n if n else None},
        'prior_click_bands_observed_only':bands,
        'reference_input_sha256':hashlib.sha256(pd.util.hash_pandas_object(observed[fields+['decoupling_signature']],index=False).values.tobytes()).hexdigest()}


def audit_fixed_episodes06(episodes,validation):
    rows=[]
    keys=['client_hash_id','content_hash_id']
    features=['impressions_prior28','clicks_prior28','ctr_prior28','avg_position_prior28']
    for fold in validation['folds']:
        origin=next(x for x in episodes if str(x)==fold['origin'])
        train_clients={client for t,frame in episodes.items() if str(t) in fold['training_origins']
                       for client in frame.loc[frame.decoupling_signature.notna(),'client_hash_id']}
        future=episodes[origin]
        for arm in fold['arms']:
            if arm['status']!='scored':continue
            if arm['arm']=='hash_heldout_clients':
                mask=future.client_hash_id.map(lambda client:int(hashlib.sha256(('flyrank-fix05-v1|'+str(client)).encode()).hexdigest(),16)%4==0)
            else:
                seen=future.client_hash_id.isin(train_clients)
                mask=seen if arm['arm']=='known_clients' else ~seen
            frame=future.loc[mask].sort_values(keys,kind='stable').reset_index(drop=True)
            digest=hashlib.sha256(pd.util.hash_pandas_object(frame[keys+features+['decoupling_signature']],index=False).values.tobytes()).hexdigest()
            if digest!=arm['cohort_sha256']:raise ValueError('Fix05 cohort/features/labels differ from the returned run.')
            stats=summarize_count_noise06(frame)
            if stats['full_arm_observation']['n_pages']!=arm['evaluation']['n_page_episodes']:
                raise ValueError('Evaluation population changed.')
            rows.append({'origin':str(origin),'arm':arm['arm'],'source_cohort_sha256':digest,**stats})
    return {'protocol':COUNT_PROTOCOL06,'protocol_sha256':hashlib.sha256(json.dumps(COUNT_PROTOCOL06,sort_keys=True).encode()).hexdigest(),
            'source_fix05_protocol_sha256':validation['protocol_sha256'],'rows':rows,
            'interpretation':'Reference-model diagnostic only. An observed signature is not declared spurious, and model rankings are unchanged.'}


In [ ]:
"""Fixed-queue partial identification and client-portfolio influence diagnostics."""
import hashlib,json,warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

KEYS07=['client_hash_id','content_hash_id']
FEATURES07=['impressions_prior28','clicks_prior28','ctr_prior28','avg_position_prior28']
PAIRS07=[('Logistic Regression','Prior CTR'),('Logistic Regression','Visible pooled-CTR excess')]
PROTOCOL07={
    'version':'fix07-v1','status':'exploratory_paired_diagnostic_after_Fix06_review',
    'population':'Unchanged Fix05 known-client as-of eligible frames; no total-click cap or future-outcome filtering.',
    'origins':['2026-02-01','2026-03-01','2026-04-01','2026-05-01'],
    'budget':50,'pairs':[list(pair) for pair in PAIRS07],
    'replay':'Four fits of unchanged LR settings recover the original predictions. Require exact prediction hashes for LR, prior CTR and pooled CTR excess.',
    'models':'No new candidate, tuning or Random-Forest fit. All eight original Fix05 results remain retained.',
    'tie_policy':'Stable descending scores in original canonical client/content order.',
    'pair_bound':'Let d be known-positive A-only minus B-only. Bounds on P50(A)-P50(B) are (d-unknown_B_only)/50 and (d+unknown_A_only)/50.',
    'overlap':'Shared pages, including shared unknown outcomes, cancel exactly in a paired difference.',
    'client_influence':'Delete each eligible client once, rerank fixed predictions on remaining pages at the same K. Do not refit after deletion.',
    'limitations':['Missing-data bounds apply to these finite queues under the frozen tie policy; they are not confidence intervals.',
                   'Client deletion measures evaluation-portfolio sensitivity, not uncertainty of training or unseen-client model performance.',
                   'Repeated clients/pages and only two later periods limit generalization. No iid tests or bootstrap confidence intervals.',
                   'Diagnostic comparisons are exploratory, not a new blind confirmation or a change of the primary prediction task.',
                   'Counterfactual missing labels may have additional unmodeled constraints; reported bounds allow any binary labels for exclusive unknowns.',
                   'No warehouse query or June access.']}


def paired_bounds07(frame,score_a,score_b,k=50):
    a,b=[np.asarray(s,dtype=float) for s in (score_a,score_b)]
    if any(s.ndim!=1 or len(s)!=len(frame) or not np.isfinite(s).all() for s in (a,b)):
        raise ValueError('Scores must be finite, aligned one-dimensional arrays.')
    if k<1:raise ValueError('Review budget must be positive.')
    if len(frame)<k:return {'status':'insufficient_candidates','budget':k,'n_candidates':len(frame)}
    if not frame.decoupling_signature.dropna().isin([0,1]).all():raise ValueError('Labels must be binary or unknown.')
    qa=np.zeros(len(frame),dtype=bool);qb=qa.copy()
    qa[np.argsort(-a,kind='stable')[:k]]=True
    qb[np.argsort(-b,kind='stable')[:k]]=True
    known=frame.decoupling_signature.notna().to_numpy()
    positive=frame.decoupling_signature.eq(1).fillna(False).to_numpy(dtype=bool)
    only_a=qa&~qb;only_b=qb&~qa;shared=qa&qb
    ha=int((only_a&positive).sum());hb=int((only_b&positive).sum())
    ua=int((only_a&~known).sum());ub=int((only_b&~known).sum())
    d=ha-hb
    return {'status':'evaluated','budget':k,'n_candidates':len(frame),
        'overlap_pages':int(shared.sum()),'jaccard':float(shared.sum()/(qa|qb).sum()),
        'shared_observed_positive_count':int((shared&positive).sum()),
        'shared_unknown_count':int((shared&~known).sum()),
        'a_only_pages':int(only_a.sum()),'b_only_pages':int(only_b.sum()),
        'a_only_observed_positives':ha,'b_only_observed_positives':hb,
        'a_only_unknown_count':ua,'b_only_unknown_count':ub,
        'a_observed_hits':int((qa&positive).sum()),'b_observed_hits':int((qb&positive).sum()),
        'a_unknown_count':int((qa&~known).sum()),'b_unknown_count':int((qb&~known).sum()),
        'known_contribution_difference':d/k,
        'paired_precision_difference_lower':(d-ub)/k,
        'paired_precision_difference_upper':(d+ua)/k,
        'point_difference_if_exclusive_outcomes_observed':d/k if ua+ub==0 else None}


def replay_scores07(train,test):
    # Exactly the four-feature LR construction used by score_models05.
    model=make_pipeline(StandardScaler(),LogisticRegression(C=1.0,max_iter=1000,random_state=42))
    with warnings.catch_warnings(),threadpool_limits(limits=1):
        warnings.simplefilter('error',ConvergenceWarning)
        model.fit(train[FEATURES07].to_numpy(dtype=float),train.decoupling_signature.to_numpy(dtype=int))
        learned=model.predict_proba(test[FEATURES07].to_numpy(dtype=float))[:,1]
    rules,_=score_transparent_baselines(fit_pooled_reference(train),test)
    return {'Logistic Regression':learned,'Prior CTR':rules['Prior CTR'],
            'Visible pooled-CTR excess':rules['Visible pooled-CTR excess']}


def client_deletion07(frame,scores_a,scores_b,aliases,k=50):
    rows=[]
    for client in sorted(frame.client_hash_id.unique()):
        keep=~frame.client_hash_id.eq(client).to_numpy()
        rows.append({'deleted_client':aliases[str(client)],'n_removed_pages':int((~keep).sum()),
                     **paired_bounds07(frame.loc[keep],np.asarray(scores_a)[keep],np.asarray(scores_b)[keep],k)})
    return rows


def run_paired_audit07(episodes,validation):
    aliases={client:f'client_{i+1:03d}' for i,client in enumerate(sorted(
        {str(client) for frame in episodes.values() for client in frame.client_hash_id}))}
    origins=[]
    for fold in validation['folds']:
        arm=next(a for a in fold['arms'] if a['arm']=='known_clients')
        if arm['status']!='scored':raise ValueError('Expected the reviewed scored known-client arm.')
        lookup={str(t):frame for t,frame in episodes.items()}
        history=[lookup[t].sort_values(KEYS07,kind='stable').reset_index(drop=True) for t in fold['training_origins']]
        if len(history)!=3:raise ValueError('Three original training origins are required.')
        train=pd.concat(history,ignore_index=True)
        train=train.loc[train.decoupling_signature.notna()].copy()
        future=next(frame for t,frame in episodes.items() if str(t)==fold['origin'])
        test=future.loc[future.client_hash_id.isin(set(train.client_hash_id))].sort_values(KEYS07,kind='stable').reset_index(drop=True)
        digest=hashlib.sha256(pd.util.hash_pandas_object(test[KEYS07+FEATURES07+['decoupling_signature']],index=False).values.tobytes()).hexdigest()
        if digest!=arm['cohort_sha256']:raise ValueError('Fix05 known-client cohort/features/labels changed.')
        scores=replay_scores07(train,test)
        hashes={name:hashlib.sha256(np.asarray(score,dtype=float).tobytes()).hexdigest() for name,score in scores.items()}
        if any(value!=arm['prediction_sha256'][name] for name,value in hashes.items()):
            raise ValueError('Replayed predictions do not exactly match Fix05. Stop; do not reinterpret a changed ranking.')
        pairs=[]
        for a,b in PAIRS07:
            base=paired_bounds07(test,scores[a],scores[b])
            for name,prefix in [(a,'a'),(b,'b')]:
                original=next(row for row in arm['metrics'] if row['method']==name and row['minimum_prior_clicks']==1)
                q=next(q for q in original['queues'] if q['budget']==50)
                if (base[prefix+'_observed_hits'],base[prefix+'_unknown_count'])!=(q['observed_hits'],q['unknown_outcomes']):
                    raise ValueError('Original queue hit/unknown accounting differs after replay.')
            deletion=client_deletion07(test,scores[a],scores[b],aliases)
            valid=[row for row in deletion if row['status']=='evaluated']
            pairs.append({'method_a':a,'method_b':b,'full_portfolio':base,
                'client_deletions':deletion,
                'influence_summary':{'n_clients':len(deletion),'n_supported_deletions':len(valid),
                    'minimum_lower_bound_after_deletion':min((x['paired_precision_difference_lower'] for x in valid),default=None),
                    'maximum_upper_bound_after_deletion':max((x['paired_precision_difference_upper'] for x in valid),default=None),
                    'n_deletions_with_lower_bound_above_zero':sum(x['paired_precision_difference_lower']>0 for x in valid),
                    'n_deletions_with_upper_bound_below_zero':sum(x['paired_precision_difference_upper']<0 for x in valid)}})
        origins.append({'origin':fold['origin'],'arm':'known_clients','source_cohort_sha256':digest,
            'verified_prediction_sha256':hashes,'evaluation':arm['evaluation'],'pairs':pairs})
    return {'protocol':PROTOCOL07,'protocol_sha256':hashlib.sha256(json.dumps(PROTOCOL07,sort_keys=True).encode()).hexdigest(),
            'source_fix05_protocol_sha256':validation['protocol_sha256'],'origins':origins}


In [ ]:
"""Development analysis entry point. Public exports contain aggregate results only."""
from datetime import datetime, timezone
from pathlib import Path
import hashlib, importlib, importlib.metadata, json, os, platform, re
import duckdb

PINS = {'numpy':'2.2.6','pandas':'2.2.3','scipy':'1.16.3',
    'scikit-learn':'1.7.2','matplotlib':'3.10.7','duckdb':'1.4.1','huggingface_hub':'1.0.1'}
SOURCE_REVISION = '50cbf7c3909d07be4d1b5906b4d09e882e5acbf2'
LEGACY_EXTRACTOR = '7b1c93304051641d59bd35b7fcb8bfb5dc85c54571b36c4736eb41453d1e71cd'
LEGACY_CACHE = Path('work/private_cache/fix05/memory_recovery_v1/windows.parquet')
LEGACY_CACHE_SHA = 'f85b104bc9281ee54903c025a5c835699dc98a8c5afd02f72e669594f09d2d4c'


def digest_json(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True).encode()).hexdigest()


def verify_versions():
    errors=[]
    for name,expected in PINS.items():
        module=importlib.import_module('sklearn' if name=='scikit-learn' else name)
        loaded=getattr(module,'__version__',None)
        installed=importlib.metadata.version(name)
        if loaded!=expected or installed!=expected:
            errors.append(f'{name}: loaded={loaded}, installed={installed}, expected={expected}')
    if errors:
        raise RuntimeError('Restart the Colab session after setup, then Run all. '+ '; '.join(errors))


def development_protocol():
    return {'version':'clean-development-v1',
        'status':'consolidated_retrospective_analysis_not_preregistration',
        'forecast_protocol':PROTOCOL05,'forecast_protocol_sha256':protocol_hash05(),
        'count_reference_protocol':COUNT_PROTOCOL06,'paired_diagnostic_protocol':PROTOCOL07,
        'june_read':False,'june_prior_user_exposure':'uncertain',
        'purpose':'Reproduce repaired February-May analyses without continuation-cell dependencies.',
        'interpretation':'No blind-test, causal, equivalence or population-wide ML-failure claim.'}


def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,allow_nan=False)+'\n',encoding='utf-8')


def verified_cache(path,extractor_sha,synthetic=False):
    """Validate provenance and bytes before loading any page-key aggregates."""
    path=Path(path)
    receipt_path=path.with_suffix('.json')
    if not receipt_path.exists():raise RuntimeError('Cache receipt is missing; preserve files and investigate.')
    receipt=json.loads(receipt_path.read_text(encoding='utf-8'))
    producers={extractor_sha} if synthetic else {extractor_sha,LEGACY_EXTRACTOR}
    if receipt.get('implementation_sha256') not in producers:
        raise RuntimeError('Unrecognized cache producer; do not mix extraction implementations.')
    expected={'protocol_sha256':protocol_hash05(),'source_months':MONTHS05,
        'kind':'combined_verified_windows',
        'dataset_revision':'synthetic_fixture_only' if synthetic else SOURCE_REVISION}
    if any(receipt.get(k)!=v for k,v in expected.items()):
        raise RuntimeError('Cache source, date scope or protocol differs from the fixed development study.')
    checked_checkpoint05m(path,{**expected,'implementation_sha256':receipt['implementation_sha256']})
    if not synthetic and receipt['implementation_sha256']==LEGACY_EXTRACTOR and receipt['sha256']!=LEGACY_CACHE_SHA:
        raise RuntimeError('Legacy cache bytes differ from the reviewed warehouse run.')
    return receipt


def acquire_development_cache(con,extractor_sha,synthetic=False,cache_path=None,progress=print):
    root=Path('work/private_cache/study/development')/extractor_sha[:16]
    candidate=Path(cache_path) if cache_path is not None else (LEGACY_CACHE if LEGACY_CACHE.exists() else root/'windows.parquet')
    if candidate.exists():
        receipt=verified_cache(candidate,extractor_sha,synthetic)
        progress('Verified local cache reused; no warehouse access.')
        return candidate,receipt
    if synthetic:raise RuntimeError('Synthetic mode requires a prepared private fixture cache; remote access is disabled.')
    if cache_path is not None:raise RuntimeError('Explicit cache path does not exist; no remote fallback.')
    api=None
    def revision():
        nonlocal api
        if api is None:
            token=os.environ.get('HF_TOKEN')
            if not token:
                try:
                    from google.colab import userdata
                    token=userdata.get('HF_TOKEN')
                except Exception:token=None
            if not token or not re.fullmatch(r'hf_[A-Za-z0-9]+',token):
                raise RuntimeError('Enable the HF_TOKEN Colab Secret; do not paste a token.')
            from huggingface_hub import HfApi
            api=HfApi(token=token)
            try:con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '"+token+"')")
            except Exception:raise RuntimeError('Could not configure private warehouse access.') from None
        value=api.dataset_info('FlyRank/internship-warehouse').sha
        if value!=SOURCE_REVISION:raise RuntimeError('Warehouse release changed; do not combine releases.')
        return value
    def sources(month):
        if month not in MONTHS05:raise ValueError('Month outside the fixed development protocol.')
        return ['hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month='+month+'/*.parquet']
    return build_cache05m(con,windows05(),MONTHS05,root,protocol_hash05(),extractor_sha,sources,revision,progress)


def run_development(extractor_sha,implementation_sha,*,cache_path=None,synthetic=False,
                    output_root=None,progress=print):
    """Freeze receipt, load/build verified cache, score, diagnose, export aggregates."""
    verify_versions()
    for value in (extractor_sha,implementation_sha):
        if not re.fullmatch('[a-f0-9]{64}',value):raise ValueError('Expected SHA256 implementation identity.')
    if protocol_hash05()!='fa08ab634857e99ecdd9b194c93105795258b31ddaed49454c55f2d8a478e0a8':
        raise RuntimeError('Forecast protocol changed before data access.')
    protocol=development_protocol()
    root=Path(output_root or ('work/research/results/synthetic' if synthetic else 'work/research/results/development'))
    run=root/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    run.mkdir(parents=True,exist_ok=False)
    write_json(run/'protocol.json',{'protocol':protocol,'protocol_sha256':digest_json(protocol),
        'created_utc':datetime.now(timezone.utc).isoformat(),'implementation_sha256':implementation_sha})
    progress('Development protocol saved before cache access. June remains excluded.')
    con=duckdb.connect()
    try:
        settings=memory_settings05m(con,Path('work/private_cache/study/spill'))
        cache,receipt=acquire_development_cache(con,extractor_sha,synthetic,cache_path,progress)
        validate_aggregate05m(con,cache,merged=True)
        windows=con.execute('SELECT DISTINCT origin,role FROM '+parquet05m([cache])).fetchall()
        expected={(str(t),r) for t in EPISODES05 for r in ('feature','outcome')}
        if set(windows)!=expected:raise RuntimeError('Cached window roster differs from the fixed fourteen windows.')
        episodes={};audits=[]
        for origin in EPISODES05:
            aggregate=con.execute('SELECT * FROM '+parquet05m([cache])+' WHERE origin=?',[str(origin)]).df()
            episodes[origin],audit=build_episode05(aggregate,origin);audits.append(audit)
            del aggregate
    finally:con.close()
    progress('Private aggregate checks passed. Fitting fixed historical models.')
    validation=run_chronological05(episodes,audits,progress=progress)
    execution={'mode':'synthetic_fixture_only' if synthetic else 'user_colab_warehouse',
        'implementation_sha256':implementation_sha,'extractor_sha256':extractor_sha,
        'source':receipt,'versions':PINS,'python':platform.python_version(),
        'memory_settings':settings,'june_read':False,'raw_identifiers_or_scores_exported':False}
    validation['execution']=execution
    write_json(run/'chronological_validation.json',validation)
    progress('Auditing conditional count allocation on unchanged observed cohorts.')
    noise=audit_fixed_episodes06(episodes,validation)
    write_json(run/'count_noise_audit.json',noise)
    progress('Verifying paired LR/rule prediction hashes and client deletions.')
    paired=run_paired_audit07(episodes,validation)
    write_json(run/'paired_queue_audit.json',paired)
    manifest={'status':'complete','protocol_sha256':digest_json(protocol),'execution':execution,
        'files':{p.name:file_hash05m(p) for p in sorted(run.glob('*.json'))},
        'scored_arms':sum(a['status']=='scored' for f in validation['folds'] for a in f['arms']),
        'count_reference_rows':len(noise['rows']),
        'paired_comparisons':sum(len(o['pairs']) for o in paired['origins'])}
    write_json(run/'run_manifest.json',manifest)
    progress('COMPLETE: aggregate results saved in '+str(run))
    return run


In [ ]:
"""Additional retrospective June evaluation; definitions do not access data."""
from datetime import date,datetime,timezone
from pathlib import Path
import hashlib,json,os,re,uuid
import numpy as np
import pandas as pd
import duckdb

JUNE_ORIGIN=date(2026,6,1)
JUNE_TRAINING=[date(2026,m,1) for m in (3,4,5)]
JUNE_PROTOCOL_SHA='cfbee182ac361efe2b932d9131f18c62dcb2694e3038a2ff4234df018ecf4c07'
JUNE_ARMS=['known_clients','naturally_unseen_clients','hash_heldout_clients']


def june_windows():
    return pd.DataFrame([{'origin':'2026-06-01','role':'outcome',
        'start_date':date(2026,6,1),'end_date':date(2026,6,28)}])


def june_source_paths():
    return ['hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-06/*.parquet']


def june_feature_from_may(aggregate):
    required=set(KEYS05+['origin','role','raw_rows','raw_days','available_days',
        'bad_counts','bad_position','impressions','clicks','sum_position'])
    if not required.issubset(aggregate.columns):
        raise ValueError('June features require the unfiltered raw May window aggregate, not a May episode frame.')
    if set(aggregate.origin.astype(str))!={'2026-05-01'} or set(aggregate.role)!={'outcome'}:
        raise ValueError('Only the unfiltered May 1-28 outcome window can become June features.')
    result=aggregate.copy()
    result['origin']='2026-06-01';result['role']='feature'
    return result


def frame_sha_june(frame,labels=False):
    columns=KEYS05+FEATURES05+(['decoupling_signature'] if labels else [])
    return hashlib.sha256(pd.util.hash_pandas_object(frame[columns],index=False).values.tobytes()).hexdigest()


def scores_sha_june(scores):
    return {name:hashlib.sha256(np.asarray(scores[name],dtype=float).tobytes()).hexdigest() for name in METHODS05}


def training_and_candidates_june(episodes,asof):
    if train_origins05(JUNE_ORIGIN)!=JUNE_TRAINING:
        raise RuntimeError('June maturity schedule changed.')
    if any(t not in episodes for t in JUNE_TRAINING):
        raise RuntimeError('Exactly the March, April and May episodes are required.')
    train=pd.concat([episodes[t] for t in JUNE_TRAINING],ignore_index=True)
    train=train.loc[train.decoupling_signature.notna()].copy()
    if not asof.decoupling_signature.isna().all():
        raise ValueError('Predictions must be frozen before June outcomes are attached.')
    result=[]
    for arm in JUNE_ARMS:
        tr=train.copy()
        if arm=='hash_heldout_clients':
            tr=tr.loc[~tr.client_hash_id.map(heldout05)]
            test=asof.loc[asof.client_hash_id.map(heldout05)].copy()
            if set(tr.client_hash_id)&set(test.client_hash_id):raise AssertionError('Hash client leakage.')
        else:
            seen=asof.client_hash_id.isin(set(tr.client_hash_id))
            test=asof.loc[seen if arm=='known_clients' else ~seen].copy()
        test=test.sort_values(KEYS05,kind='stable').reset_index(drop=True)
        record={'arm':arm,'training':cohort_summary05(tr),'n_as_of_candidates':len(test),
            'candidate_feature_sha256':frame_sha_june(test),
            'training_frame_sha256':frame_sha_june(tr,labels=True)}
        counts=tr.decoupling_signature.value_counts()
        if set(tr.origin)!={str(t) for t in JUNE_TRAINING}:status='insufficient_training_history'
        elif len(tr)<500 or tr.client_hash_id.nunique()<2 or min(counts.get(0,0),counts.get(1,0))<20:
            status='insufficient_training_support'
        elif not len(test):status='no_eligible_test_pages'
        else:status='scored'
        record['status']=status
        result.append((record,tr,test))
    return result


def freeze_predictions_june(con,episodes,asof,path,identity,progress=print):
    """Private durable prediction data, public-safe checksum receipt; no June input."""
    plan=training_and_candidates_june(episodes,asof)
    base={**identity,'kind':'june_predictions_before_outcomes',
        'training_origins':[str(t) for t in JUNE_TRAINING],
        'candidate_feature_sha256':frame_sha_june(asof),
        'arm_inputs':[record for record,_,_ in plan], 'methods':METHODS05,
        'versions':PINS,'june_outcomes_accessed_before_prediction_freeze':False}
    path=Path(path)
    ready=checked_checkpoint05m(path,base)
    if ready:
        progress('Verified private June predictions reused; no model refitting.')
        packed=con.execute('SELECT * FROM '+parquet05m([path])).df()
        records=ready['arms']
    else:
        parts=[];records=[]
        for record,tr,test in plan:
            record=dict(record)
            if record['status']=='scored':
                progress('Fitting June '+record['arm']+' before accessing June outcomes.')
                scores,audit=score_models05(tr,test)
                record.update(reference_audit=audit,prediction_sha256=scores_sha_june(scores))
                part=test[KEYS05+FEATURES05].copy();part['arm']=record['arm']
                for method in METHODS05:part[method]=scores[method]
                parts.append(part)
            records.append(record)
        packed=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame(columns=KEYS05+FEATURES05+['arm']+METHODS05)
        con.register('_june_private_predictions',packed)
        ready=write_checkpoint05m(con,'SELECT * FROM _june_private_predictions',path,
            {**base,'arms':records,'frozen_utc':datetime.now(timezone.utc).isoformat()},validate=False)
    restored={}
    if packed.duplicated(['arm']+KEYS05).any():raise RuntimeError('Duplicate frozen candidate scores.')
    for record,_,test in plan:
        saved=next(r for r in records if r['arm']==record['arm'])
        if any(saved.get(k)!=v for k,v in record.items()):raise RuntimeError('Frozen arm differs from its historical input plan.')
        if record['status']!='scored':continue
        part=packed.loc[packed.arm.eq(record['arm'])].sort_values(KEYS05,kind='stable').reset_index(drop=True)
        for column in ['impressions_prior28','clicks_prior28']:part[column]=part[column].astype('Int64')
        scores={name:part[name].to_numpy(dtype=float) for name in METHODS05}
        if frame_sha_june(part)!=record['candidate_feature_sha256'] or scores_sha_june(scores)!=saved['prediction_sha256']:
            raise RuntimeError('Frozen candidate alignment or prediction bytes changed.')
        restored[record['arm']]=(test,scores)
    if set(packed.arm)-{r['arm'] for r in records if r['status']=='scored'}:
        raise RuntimeError('Unexpected arm in private predictions.')
    return records,restored,ready


def acquire_june_outcomes(con,root,identity,*,synthetic_source=None,prediction_frozen_utc=None,progress=print):
    """One June partition only; retries reuse a verified local checkpoint."""
    root=Path(root);path=root/'outcomes.parquet'
    expected={**identity,'kind':'june_outcome_window','source_month':'2026-06',
        'start_date':'2026-06-01','end_date':'2026-06-28'}
    ready=checked_checkpoint05m(path,expected)
    if ready:
        if 'first_access_attempt_utc' not in ready:
            raise RuntimeError('June checkpoint lacks its original access-order receipt.')
        if prediction_frozen_utc is not None and datetime.fromisoformat(prediction_frozen_utc)>datetime.fromisoformat(ready['first_access_attempt_utc']):
            raise RuntimeError('Predictions were frozen after an earlier June access attempt; do not reuse this outcome checkpoint.')
        progress('Verified June outcome checkpoint reused; no remote query.')
        validate_aggregate05m(con,path)
        return path,ready,False
    # Shared across code-version directories in this VM, so a changed wrapper
    # cannot silently create a new "first" access after June was already read.
    ledger_path=root.parent/'first_access_attempt.json'
    ledger_identity={'protocol_sha256':identity['protocol_sha256'],
        'dataset_revision':identity['dataset_revision'],
        'kind':'first_June_outcome_acquisition_attempt_across_implementations'}
    if ledger_path.exists():
        ledger=json.loads(ledger_path.read_text(encoding='utf-8'))
        if any(ledger.get(k)!=v for k,v in ledger_identity.items()):
            raise RuntimeError('June first-access ledger identity differs; preserve it and investigate.')
    else:
        ledger={**ledger_identity,'first_access_attempt_utc':datetime.now(timezone.utc).isoformat()}
        write_json(ledger_path,ledger)
    if prediction_frozen_utc is not None and datetime.fromisoformat(prediction_frozen_utc)>datetime.fromisoformat(ledger['first_access_attempt_utc']):
        raise RuntimeError('Predictions were frozen after an earlier June access attempt; stop for review.')
    expected={**expected,'first_access_attempt_utc':ledger['first_access_attempt_utc']}
    synthetic=identity['dataset_revision']=='synthetic_fixture_only'
    if synthetic:
        if synthetic_source is None:raise RuntimeError('Synthetic June run requires a local daily fixture; remote access is disabled.')
        sources=[str(synthetic_source)];revision=lambda:'synthetic_fixture_only'
    else:
        if synthetic_source is not None:raise ValueError('A synthetic source cannot replace warehouse outcomes.')
        token=os.environ.get('HF_TOKEN')
        if not token:
            try:
                from google.colab import userdata
                token=userdata.get('HF_TOKEN')
            except Exception:token=None
        if not token or not re.fullmatch(r'hf_[A-Za-z0-9]+',token):
            raise RuntimeError('Enable the approved HF_TOKEN Colab Secret; never paste a token.')
        from huggingface_hub import HfApi
        api=HfApi(token=token)
        try:con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '"+token+"')")
        except Exception:raise RuntimeError('Could not configure private warehouse access.') from None
        del token
        revision=lambda:api.dataset_info('FlyRank/internship-warehouse').sha
        sources=june_source_paths()
    if revision()!=identity['dataset_revision']:raise RuntimeError('June release differs from verified historical cache.')
    con.register('_fix05_windows',june_windows())
    progress('Aggregating only June 1-28 from the June partition; predictions are already frozen.')
    stage=root/'pending'/('outcomes-'+uuid.uuid4().hex+'.parquet')
    metadata=write_checkpoint05m(con,month_sql05m(sources,'2026-06'),stage,expected)
    if revision()!=identity['dataset_revision']:
        raise RuntimeError('Warehouse revision changed during June aggregation; pending checkpoint is not approved.')
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():path.replace(path.with_name('outcomes.unverified-'+uuid.uuid4().hex+'.parquet'))
    stage.replace(path);stage.with_suffix('.json').replace(path.with_suffix('.json'))
    return path,metadata,not synthetic


def report_june(episodes,feature,outcome,records,frozen,protocol):
    evaluated,audit=build_episode05(pd.concat([feature,outcome],ignore_index=True),JUNE_ORIGIN)
    asof,_=build_episode05(feature,JUNE_ORIGIN)
    if frame_sha_june(evaluated)!=frame_sha_june(asof):raise RuntimeError('Outcome attachment changed June candidates or features.')
    aliases={str(client):f'client_{i+1:03d}' for i,client in enumerate(sorted(evaluated.client_hash_id.unique()))}
    arms=[];paired_rows=[];noise_rows=[]
    for saved in records:
        row={**saved,'metrics':[]};arms.append(row)
        if saved['status']!='scored':continue
        before,scores=frozen[saved['arm']]
        # Explicit one-to-one key alignment, preserving the frozen ordering.
        test=before[KEYS05].merge(evaluated,on=KEYS05,how='left',validate='one_to_one',sort=False)
        if frame_sha_june(test)!=saved['candidate_feature_sha256']:
            raise RuntimeError('June labels are not aligned to frozen predictions.')
        row['evaluation']=cohort_summary05(test)
        row['cohort_sha256']=frame_sha_june(test,labels=True)
        for minimum in (1,5,20):
            mask=test.clicks_prior28.ge(minimum).to_numpy(dtype=bool)
            for name,score in scores.items():
                row['metrics'].append({'method':name,'minimum_prior_clicks':minimum,
                    'support_population':cohort_summary05(test.loc[mask]),**metrics05(test.loc[mask],score[mask])})
        details=[]
        for name,score in scores.items():
            selected=test.iloc[np.argsort(-score,kind='stable')[:50]] if len(test)>=50 else test.iloc[:0]
            for client,part in test.groupby('client_hash_id',sort=True):
                queue=selected.loc[selected.client_hash_id.eq(client)]
                details.append({'anonymous_client':aliases[str(client)],'method':name,**cohort_summary05(part),
                    'global_queue_pages':len(queue),'global_queue_observed_hits':int(queue.decoupling_signature.sum()),
                    'global_queue_unknown':int(queue.decoupling_signature.isna().sum())})
        row['anonymous_client_details']=details
        pairs=[]
        for a,b in PAIRS07:
            deletion=client_deletion07(test,scores[a],scores[b],aliases)
            valid=[x for x in deletion if x['status']=='evaluated']
            pairs.append({'method_a':a,'method_b':b,
                'role':'primary' if saved['arm']=='known_clients' and b=='Prior CTR' else 'secondary_descriptive',
                'full_portfolio':paired_bounds07(test,scores[a],scores[b]),'client_deletions':deletion,
                'influence_summary':{'n_clients':len(deletion),'n_supported_deletions':len(valid),
                    'minimum_lower_bound_after_deletion':min((x['paired_precision_difference_lower'] for x in valid),default=None),
                    'maximum_upper_bound_after_deletion':max((x['paired_precision_difference_upper'] for x in valid),default=None)}})
        paired_rows.append({'origin':'2026-06-01','arm':saved['arm'],'pairs':pairs,
            'source_cohort_sha256':row['cohort_sha256']})
        noise_rows.append({'origin':'2026-06-01','arm':saved['arm'],**summarize_count_noise06(test)})
    validation={'protocol':protocol,'protocol_sha256':JUNE_PROTOCOL_SHA,'cohort_audits':[audit],
        'folds':[{'origin':'2026-06-01','cutoff':'2026-05-28','training_origins':[str(t) for t in JUNE_TRAINING],
            'status':'completed_with_explicit_arm_statuses','arms':arms}],
        'limitations':protocol['limits']}
    paired={'source_protocol_sha256':JUNE_PROTOCOL_SHA,'direction':'LR precision@50 minus comparator precision@50',
        'interpretation':'Fixed-queue logical bounds, not confidence intervals. Client deletion uses fixed scores without refitting.',
        'anonymous_alias_scope':'This June run only; do not infer cross-run alias identity.','rows':paired_rows}
    count_protocol={**COUNT_PROTOCOL06,'version':'june-count-reference-v1',
        'status':'secondary_descriptive_diagnostic_after_frozen_June_prediction',
        'source':'Historical training plus June feature/outcome aggregates under the June addendum; never used for prediction.',
        'population':'Observed members of each June fixed as-of arm, restricted to total clicks <=200.'}
    noise={'protocol':count_protocol,'protocol_sha256':digest_json(count_protocol),
        'source_protocol_sha256':JUNE_PROTOCOL_SHA,'rows':noise_rows}
    return validation,paired,noise


def run_june(protocol_receipt,extractor_sha,implementation_sha,*,cache_path=None,synthetic=False,
             synthetic_source=None,output_root=None,progress=print):
    verify_versions()
    protocol=protocol_receipt['protocol']
    if digest_json(protocol)!=JUNE_PROTOCOL_SHA or protocol_receipt['protocol_sha256']!=JUNE_PROTOCOL_SHA:
        raise RuntimeError('June protocol differs from the version committed before access.')
    for value in (extractor_sha,implementation_sha):
        if not re.fullmatch('[a-f0-9]{64}',value):raise ValueError('Expected implementation SHA256.')
    root=Path(output_root or ('work/research/results/synthetic_june' if synthetic else 'work/research/results/june'))
    run=root/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ');run.mkdir(parents=True,exist_ok=False)
    write_json(run/'protocol.json',protocol_receipt)
    write_json(run/'execution_plan.json',{'created_utc':datetime.now(timezone.utc).isoformat(),
        'protocol_commit':'6cba97f','protocol_sha256':JUNE_PROTOCOL_SHA,'implementation_sha256':implementation_sha,
        'mode':'synthetic_fixture_only' if synthetic else 'user_colab_warehouse',
        'prior_June_user_exposure':'uncertain','June_outcome_access_in_this_run_before_execution_plan':False,
        'previous_workflow_access_status_at_plan_creation':'not_assessed; checked against the immutable access ledger later'})
    progress('Committed June protocol and execution identity saved before cache or source access.')
    private=Path('work/private_cache/study/june_synthetic' if synthetic else 'work/private_cache/study/june')/implementation_sha[:16]
    candidate=Path(cache_path) if cache_path else (LEGACY_CACHE if LEGACY_CACHE.exists() else
        Path('work/private_cache/study/development')/extractor_sha[:16]/'windows.parquet')
    if not candidate.exists():raise RuntimeError('Verified October-May cache is missing. Restore it or run development cache preparation first; no automatic historical rescan.')
    historical=verified_cache(candidate,extractor_sha,synthetic)
    identity={'protocol_sha256':JUNE_PROTOCOL_SHA,'implementation_sha256':implementation_sha,
        'historical_cache_sha256':historical['sha256'],'dataset_revision':historical['dataset_revision']}
    con=duckdb.connect()
    try:
        settings=memory_settings05m(con,private)
        validate_aggregate05m(con,candidate,merged=True)
        episodes={}
        for origin in JUNE_TRAINING:
            aggregate=con.execute('SELECT * FROM '+parquet05m([candidate])+' WHERE origin=?',[str(origin)]).df()
            episodes[origin],_=build_episode05(aggregate,origin)
        may=con.execute('SELECT * FROM '+parquet05m([candidate])+" WHERE origin='2026-05-01' AND role='outcome'").df()
        feature=june_feature_from_may(may);del may,aggregate
        asof,_=build_episode05(feature,JUNE_ORIGIN)
        records,frozen,prediction_receipt=freeze_predictions_june(con,episodes,asof,private/'predictions.parquet',identity,progress)
        write_json(run/'predictions_frozen_receipt.json',prediction_receipt)
        progress('All supported June rankings are frozen. No June outcomes used in fitting or candidate selection.')
        outcome_path,outcome_receipt,queried=acquire_june_outcomes(con,private,identity,
            synthetic_source=synthetic_source,prediction_frozen_utc=prediction_receipt['frozen_utc'],progress=progress)
        outcome=con.execute('SELECT * FROM '+parquet05m([outcome_path])).df()
        if len(outcome) and (set(outcome.origin.astype(str))!={'2026-06-01'} or set(outcome.role)!={'outcome'}):
            raise RuntimeError('Unexpected June outcome window roster.')
    finally:con.close()
    validation,paired,noise=report_june(episodes,feature,outcome,records,frozen,protocol)
    order_verified=datetime.fromisoformat(prediction_receipt['frozen_utc'])<=datetime.fromisoformat(outcome_receipt['first_access_attempt_utc'])
    if not order_verified:raise RuntimeError('Prediction freeze does not precede the first recorded June acquisition attempt.')
    execution={'mode':'synthetic_fixture_only' if synthetic else 'user_colab_warehouse',
        'implementation_sha256':implementation_sha,'historical_source':historical,'june_source':outcome_receipt,
        'june_partition_queried_this_run':queried,'predictions_frozen_before_June_outcome_access':order_verified,
        'prediction_frozen_utc':prediction_receipt['frozen_utc'],
        'first_June_access_attempt_utc':outcome_receipt['first_access_attempt_utc'],
        'access_order_scope':'This Colab VM and addendum across implementation checkpoints; no claim about exposure before this VM or a blind test.',
        'source_months_new':['2026-06'],'final_month_sample_used':False,'versions':PINS,
        'memory_settings':settings,'prior_user_exposure':'uncertain','independent_prospective_test':False,
        'private_identifiers_or_prediction_arrays_exported':False}
    validation['execution']=execution
    for name,value in [('june_validation.json',validation),('paired_queue_audit.json',paired),('count_noise_audit.json',noise)]:
        write_json(run/name,value)
    write_json(run/'run_manifest.json',{'status':'complete','execution':execution,
        'files':{p.name:file_hash05m(p) for p in sorted(run.glob('*.json'))},
        'scored_arms':sum(r['status']=='scored' for r in records)})
    progress('JUNE COMPLETE: aggregate-only results saved in '+str(run))
    return run


In [ ]:
"""June figure drawn from aggregate JSON only."""
from pathlib import Path
import json


def plot_june_results(run):
    import matplotlib
    matplotlib.use('Agg',force=True)
    import matplotlib.pyplot as plt
    run=Path(run)
    result=json.loads((run/'june_validation.json').read_text(encoding='utf-8'))
    paired=json.loads((run/'paired_queue_audit.json').read_text(encoding='utf-8'))
    arm=next(a for a in result['folds'][0]['arms'] if a['arm']=='known_clients')
    fig,axes=plt.subplots(1,2,figsize=(12,4.5),layout='constrained')
    if arm['status']=='scored':
        rows=[r for r in arm['metrics'] if r['minimum_prior_clicks']==1]
        for i,row in enumerate(rows):
            queue=next(q for q in row['queues'] if q['budget']==50)
            if queue.get('reviewed')!=50:continue
            low=100*queue['precision_lower_bound'];high=100*queue['precision_upper_bound']
            axes[0].plot([low,high],[i,i],color='#0072B2',lw=3,marker='|')
        axes[0].set_yticks(range(len(rows)),[r['method'] for r in rows])
        axes[0].invert_yaxis()
        pair=next(r for r in paired['rows'] if r['arm']=='known_clients')
        for i,row in enumerate(pair['pairs']):
            bound=row['full_portfolio']
            if bound['status']=='evaluated':
                axes[1].plot([100*bound['paired_precision_difference_lower'],100*bound['paired_precision_difference_upper']],
                    [i,i],color='#D55E00',lw=3,marker='|')
        axes[1].set_yticks([0,1],['LR minus prior CTR\n(primary)','LR minus visible CTR excess\n(secondary)'])
        axes[1].invert_yaxis()
    else:
        for ax in axes:ax.text(.5,.5,arm['status'],ha='center',transform=ax.transAxes)
    axes[0].set_title('Precision at 50: logical bounds')
    axes[0].set_xlabel('Percent of the fixed review queue')
    axes[1].set_title('Paired precision difference: logical bounds')
    axes[1].set_xlabel('Percentage points; positive favors LR')
    axes[1].axvline(0,color='0.5',linestyle='--',lw=1)
    for ax in axes:ax.grid(axis='x',alpha=.2)
    prefix='SYNTHETIC CHECK: ' if result['execution']['mode']=='synthetic_fixture_only' else ''
    fig.suptitle(prefix+'June 2026: additional retrospective evaluation, known clients')
    for extension in ['png','svg']:fig.savefig(run/('june_summary.'+extension),dpi=180)
    plt.close(fig)
    caption=('Left: precision bounds for the same fixed review budget of 50, retaining pages whose June labels '
        'are unknown. Right: LR-minus-comparator differences with shared pages canceling. These ranges are '
        'logical missing-label bounds, not confidence intervals. Only the known-client LR-minus-prior-CTR '
        'comparison is primary. All client arms, methods, coverage, secondary budgets and support checks are '
        'retained in the JSON files. Previous June exposure is uncertain; this is not a blind test.\n')
    (run/'figure_caption.txt').write_text(caption,encoding='utf-8')
    return run/'june_summary.png'


In [ ]:
JUNE_PROTOCOL_RECEIPT = {'protocol': {'version': 'june-retrospective-addendum-v2', 'supersedes_protocol_sha256': 'e7ca6f3ffae8f8fddfa7536a4c68214ea3404c4528e7c1e3df887e3bcd9ec202', 'amendment_before_June_access': 'Explicitly state paired estimand direction, primary prior-click support and training-label cutoff; no scientific settings changed.', 'status': 'protocol_prepared_before_this_workflow_accesses_June_not_preregistration', 'parent_forecast_protocol_sha256': 'fa08ab634857e99ecdd9b194c93105795258b31ddaed49454c55f2d8a478e0a8', 'prior_exposure': {'user_answer': 'I am not sure', 'status': 'uncertain', 'consequence': 'Additional retrospective validation; no blind, untouched or prospective test claim.'}, 'question': 'Does fixed logistic-regression ranking add top-50 retrieval value beyond prior CTR in an additional later period?', 'origin': '2026-06-01', 'feature_start': '2026-05-01', 'feature_end': '2026-05-28', 'gap_dates': ['2026-05-29', '2026-05-30', '2026-05-31'], 'outcome_start': '2026-06-01', 'outcome_end': '2026-06-28', 'training_origins': ['2026-03-01', '2026-04-01', '2026-05-01'], 'training_label_end_cutoff': '2026-05-28', 'feature_reuse': 'Reuse the UNFILTERED May outcome-window aggregate from the verified private cache, relabeling origin and role. Never reuse a filtered May evaluation cohort as June eligibility.', 'source_revision': '50cbf7c3909d07be4d1b5906b4d09e882e5acbf2', 'new_source_scope': 'Only fact_content_daily_performance/month=2026-06; never the sample table. Aggregate June 1-28. Reuse existing March-May episode data for training.', 'eligibility_and_label': 'Identical to parent protocol: rank all as-of eligible pages before observing outcomes; retain unknown June outcomes.', 'methods': ['Baseline rule', 'Logistic Regression', 'Random Forest', 'Prior clicks', 'Prior impressions', 'Prior CTR', 'Visible pooled-CTR shortfall', 'Visible pooled-CTR excess'], 'features': ['impressions_prior28', 'clicks_prior28', 'ctr_prior28', 'avg_position_prior28'], 'models': {'LR': {'C': 1.0, 'max_iter': 1000, 'seed': 42}, 'RF': {'n_estimators': 300, 'seed': 42, 'n_jobs': -1, 'other_parameters': 'sklearn 1.7.2 defaults'}, 'preprocessing': 'Independent StandardScaler per model, four numeric features only; no tuning or winner selection.'}, 'rules': 'Training-only position bands (0,3,10,20,50,infinity); medians or pooled CTR; visibility at training median I. High prior counts/CTR rank first.', 'arms': ['known_clients', 'naturally_unseen_clients', 'hash_heldout_clients'], 'arm_definition': 'Identical rolling-training membership and permanent hash exclusion as parent; report each arm separately.', 'primary_arm': 'known_clients', 'primary_budget': 50, 'primary_minimum_prior_clicks': 1, 'primary_comparison': ['Logistic Regression', 'Prior CTR'], 'primary_estimand': 'Delta = precision@50(Logistic Regression) minus precision@50(Prior CTR), for queues ranked from the same known-client as-of candidate frame. Positive Delta favors LR.', 'paired_bounds': 'For A=LR and B=comparator, shared pages including unknown labels cancel. Let d=observed positives in A-only minus B-only, uA=unknown outcomes in A-only and uB=unknown outcomes in B-only. Delta is bounded by (d-uB)/50 and (d+uA)/50. These logical bounds allow binary labels for exclusive unknowns and are not confidence intervals.', 'secondary_comparison': ['Logistic Regression', 'Visible pooled-CTR excess'], 'secondary_estimand': 'The same signed paired precision@50 difference, substituting visible pooled-CTR excess for B.', 'secondary_budgets': [25, 100, 200], 'secondary_minimum_prior_clicks': [5, 20], 'reporting': 'All eight methods and all three arms, cohort flow, coverage, observed hits, unknown counts and logical queue bounds. Paired missing-label bounds are primary for the specified comparison. Complete-case AUC/AP/ranking metrics are secondary.', 'unsupported_arms': 'Report the exact support/status reason. Do not shorten history or alter support thresholds.', 'no_tuning': 'No changes to labels, eligibility, features, methods, thresholds or tie order after June access. Corrections require a dated versioned amendment and disclosure.', 'diagnostics': 'Count-allocation and client-deletion analyses remain descriptive and secondary; never used to select a model or revise labels.', 'limits': ['Retrospective final data release without historical ingestion vintages.', 'Unknown prior June exposure and post-development comparator selection.', 'Repeated pages/clients and only one additional period; no iid confidence intervals.', 'Logical missing-label bounds are not confidence intervals.', 'Retrieval of the count signature does not establish business benefit from editing a page.'], 'execution_state': 'not_executed_by_this_workflow', 'implementation_requirement': 'A separate June runner must first pass date, unfiltered-feature reuse, outcome-missingness, hash exclusion and source-scope tests. The development notebook continues to exclude June.'}, 'protocol_sha256': 'cfbee182ac361efe2b932d9131f18c62dcb2694e3038a2ff4234df018ecf4c07', 'created_utc': '2026-11-04T07:54:23.135789+00:00', 'june_access_during_preparation': False}


In [ ]:
"""Execution amendment: validate June outcomes on the pre-frozen candidate universe."""
from datetime import datetime,timezone
from pathlib import Path
import json,os,re
import duckdb
import pandas as pd

ORIGINAL_JUNE_IMPLEMENTATION='39b0eccfc593ef6970e19ec3519fd6dad70e03bf9fae96b092542d0a5db4d984'
QUALITY_CONDITION_JUNE="""client_hash_id IS NULL OR content_hash_id IS NULL OR raw_rows!=raw_days OR
    raw_days!=bit_count(row_mask) OR available_days!=bit_count(available_mask) OR raw_days>28 OR
    row_mask>=268435456 OR available_days>raw_days OR (row_mask|available_mask)!=row_mask OR bad_counts>0"""


def scope_cached_june_outcomes(con,raw_path,prediction_path,asof,destination,identity):
    """No repairs/deduplication: select pre-existing candidates, then enforce all guards."""
    raw_path=Path(raw_path);prediction_path=Path(prediction_path)
    global_sha=file_hash05m(raw_path)
    keys_features=','.join(KEYS05+FEATURES05)
    union=con.execute('SELECT DISTINCT '+keys_features+' FROM '+parquet05m([prediction_path])).df()
    if union.duplicated(KEYS05).any():raise RuntimeError('Conflicting frozen features for the same candidate key.')
    for column in ['impressions_prior28','clicks_prior28']:union[column]=union[column].astype('Int64')
    union=union.sort_values(KEYS05,kind='stable').reset_index(drop=True)
    if frame_sha_june(union)!=frame_sha_june(asof):
        raise RuntimeError('Frozen score-key union differs from the full May-derived as-of universe; stop recovery.')
    con.execute('CREATE OR REPLACE TEMP VIEW _june_failed_global AS SELECT * FROM '+parquet05m([raw_path]))
    con.execute('CREATE OR REPLACE TEMP VIEW _june_frozen_keys AS SELECT DISTINCT '+','.join(KEYS05)+' FROM '+parquet05m([prediction_path]))
    columns='COUNT(*) AS n_groups,COUNT(*) FILTER(WHERE '+QUALITY_CONDITION_JUNE+') AS n_failed_groups'
    total=con.execute('SELECT '+columns+',COUNT(*) FILTER(WHERE raw_rows!=raw_days) AS duplicate_daily_key_groups,SUM(raw_rows-raw_days) AS extra_daily_rows FROM _june_failed_global').df().to_dict('records')[0]
    selected=con.execute('SELECT '+columns+' FROM _june_failed_global JOIN _june_frozen_keys USING('+','.join(KEYS05)+')').df().to_dict('records')[0]
    if selected['n_failed_groups']:
        raise RuntimeError('A source-quality failure overlaps the frozen candidate population; recovery is forbidden.')
    roster=con.execute('SELECT DISTINCT origin,role FROM _june_failed_global').fetchall()
    if set(roster)!={('2026-06-01','outcome')}:raise RuntimeError('Unexpected outcome window in the failed source artifact.')
    receipt_identity={**identity,'kind':'June_outcomes_scoped_to_pre_frozen_candidates',
        'global_failed_aggregate_sha256':global_sha,'frozen_prediction_file_sha256':file_hash05m(prediction_path),
        'full_candidate_feature_sha256':frame_sha_june(asof),
        'source_month':'2026-06','start_date':'2026-06-01','end_date':'2026-06-28'}
    ready=checked_checkpoint05m(destination,receipt_identity)
    if ready is None:
        query='SELECT a.* FROM _june_failed_global a JOIN _june_frozen_keys k USING('+','.join(KEYS05)+')'
        ready=write_checkpoint05m(con,query,destination,{**receipt_identity,
            'global_source_quality':total,'candidate_source_quality':selected,
            'data_deduplication_performed':False,'candidate_pages_removed':0,
            'original_global_aggregate_preserved':True})
    validate_aggregate05m(con,destination)
    if file_hash05m(raw_path)!=global_sha:raise RuntimeError('Global failed aggregate changed during recovery.')
    return ready


def recover_june_fixed_candidates(protocol_receipt,extractor_sha,recovery_sha,*,history_cache=None,
        original_private_root=None,synthetic=False,revision_check=None,output_root=None,progress=print):
    verify_versions()
    if protocol_receipt['protocol_sha256']!=JUNE_PROTOCOL_SHA or digest_json(protocol_receipt['protocol'])!=JUNE_PROTOCOL_SHA:
        raise RuntimeError('Preserve the committed June protocol unchanged.')
    original=Path(original_private_root or 'work/private_cache/study/june/39b0eccfc593ef69')
    candidate=Path(history_cache) if history_cache else (LEGACY_CACHE if LEGACY_CACHE.exists() else
        Path('work/private_cache/study/development')/extractor_sha[:16]/'windows.parquet')
    historical=verified_cache(candidate,extractor_sha,synthetic)
    expected_revision='synthetic_fixture_only' if synthetic else SOURCE_REVISION
    if historical['dataset_revision']!=expected_revision:raise RuntimeError('Historical release mismatch.')
    identity={'protocol_sha256':JUNE_PROTOCOL_SHA,'implementation_sha256':ORIGINAL_JUNE_IMPLEMENTATION,
        'historical_cache_sha256':historical['sha256'],'dataset_revision':expected_revision}
    prediction_path=original/'predictions.parquet'
    if not prediction_path.exists() or not prediction_path.with_suffix('.json').exists():
        raise RuntimeError('Original frozen predictions are required; recovery must never refit models.')
    before=checked_checkpoint05m(prediction_path,identity)
    ledger=json.loads((original.parent/'first_access_attempt.json').read_text(encoding='utf-8'))
    if any(ledger.get(k)!=identity[k] for k in ['protocol_sha256','dataset_revision']):
        raise RuntimeError('Original access ledger differs from the frozen study.')
    if datetime.fromisoformat(before['frozen_utc'])>datetime.fromisoformat(ledger['first_access_attempt_utc']):
        raise RuntimeError('Original predictions postdate June acquisition; stop recovery.')
    raw=list((original/'pending').glob('*.parquet'))
    if len(raw)!=1:raise RuntimeError('Expected one preserved failed June aggregate; resolve ambiguity before recovery.')
    root=Path(output_root or ('work/research/results/synthetic_june_recovery' if synthetic else 'work/research/results/june_recovery'))
    run=root/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ');run.mkdir(parents=True,exist_ok=False)
    amendment={'version':'June-candidate-source-scope-execution-amendment-v1',
        'created_utc':datetime.now(timezone.utc).isoformat(),
        'reason':'Global June source grain failed outside the frozen candidate universe.',
        'committed_protocol_sha256':JUNE_PROTOCOL_SHA,'original_implementation_sha256':ORIGINAL_JUNE_IMPLEMENTATION,
        'recovery_implementation_sha256':recovery_sha,
        'scientific_protocol_or_threshold_changes':False,'new_model_fit_or_tuning':False,
        'source_scope':'All and only the original May-derived candidate keys, before any outcome filtering.',
        'required_guards':['Exact candidate key and four-feature equality to original frozen union.',
            'Zero duplicate/count/coverage failures in candidate-scoped June outcomes.',
            'Preserve and hash failed global aggregate; no deduplication or raw-data requery.',
            'Left-join outcomes to every frozen candidate; retain absent/incomplete labels as unknown.'],
        'disclosure':'Execution-level source scoping after a global source-quality stop; not a blind test or a predeclared data-cleaning result.'}
    write_json(run/'execution_amendment.json',amendment)
    write_json(run/'protocol.json',protocol_receipt)
    if revision_check is None:
        if synthetic:raise ValueError('Synthetic recovery must provide a local revision callback.')
        token=os.environ.get('HF_TOKEN')
        if not token:
            from google.colab import userdata
            token=userdata.get('HF_TOKEN')
        if not token or not re.fullmatch(r'hf_[A-Za-z0-9]+',token):raise RuntimeError('Enable the approved Colab Secret.')
        from huggingface_hub import HfApi
        api=HfApi(token=token);del token
        revision_check=lambda:api.dataset_info('FlyRank/internship-warehouse').sha
    if revision_check()!=expected_revision:
        raise RuntimeError('June source revision changed after the original failed query; no mixed-release recovery.')
    con=duckdb.connect()
    try:
        settings=memory_settings05m(con,original/'scope_recovery')
        episodes={}
        for origin in JUNE_TRAINING:
            aggregate=con.execute('SELECT * FROM '+parquet05m([candidate])+' WHERE origin=?',[str(origin)]).df()
            episodes[origin],_=build_episode05(aggregate,origin)
        may=con.execute('SELECT * FROM '+parquet05m([candidate])+" WHERE origin='2026-05-01' AND role='outcome'").df()
        feature=june_feature_from_may(may)
        asof,_=build_episode05(feature,JUNE_ORIGIN)
        records,frozen,prediction_receipt=freeze_predictions_june(con,episodes,asof,prediction_path,identity,progress)
        scoped_path=original/'scope_recovery/candidate_outcomes.parquet'
        outcome_receipt=scope_cached_june_outcomes(con,raw[0],prediction_path,asof,scoped_path,
            {**identity,'execution_recovery_sha256':recovery_sha,
             'first_access_attempt_utc':ledger['first_access_attempt_utc']})
        outcome=con.execute('SELECT * FROM '+parquet05m([scoped_path])).df()
    finally:con.close()
    if revision_check()!=expected_revision:raise RuntimeError('Source revision changed during scoped recovery.')
    validation,paired,noise=report_june(episodes,feature,outcome,records,frozen,protocol_receipt['protocol'])
    execution={'mode':'synthetic_fixture_only' if synthetic else 'user_colab_warehouse',
        'implementation_sha256':ORIGINAL_JUNE_IMPLEMENTATION,'execution_recovery_sha256':recovery_sha,
        'historical_source':historical,'june_source':outcome_receipt,
        'june_partition_queried_this_run':False,'new_raw_warehouse_query_during_recovery':False,
        'model_fits_during_recovery':0,'source_revision_verified_after_original_query':True,
        'predictions_frozen_before_June_outcome_access':True,
        'prediction_frozen_utc':prediction_receipt['frozen_utc'],
        'first_June_access_attempt_utc':ledger['first_access_attempt_utc'],
        'source_months_new':['2026-06'],'final_month_sample_used':False,'versions':PINS,
        'memory_settings':settings,'prior_user_exposure':'uncertain','independent_prospective_test':False,
        'private_identifiers_or_prediction_arrays_exported':False,
        'source_scoping_execution_amendment':amendment['version'],
        'global_source_quality_failure_disclosed':True,'all_candidate_source_quality_guards_passed':True}
    validation['execution']=execution
    for name,value in [('june_validation.json',validation),('paired_queue_audit.json',paired),('count_noise_audit.json',noise),
        ('predictions_frozen_receipt.json',prediction_receipt)]:write_json(run/name,value)
    write_json(run/'run_manifest.json',{'status':'complete_with_disclosed_source_scoping_recovery',
        'execution':execution,'files':{p.name:file_hash05m(p) for p in sorted(run.glob('*.json'))},
        'scored_arms':sum(r['status']=='scored' for r in records)})
    progress('JUNE SCOPED RECOVERY COMPLETE: original predictions and all frozen candidates retained.')
    return run


def rerun_june_after_runtime_reset(protocol_receipt,extractor_sha,recovery_sha,*,
        history_cache=None,synthetic=False,synthetic_source=None,output_root=None,
        private_root=None,progress=print):
    """Repeat the fixed computation; never represent lost checkpoints as recovered."""
    verify_versions()
    if protocol_receipt['protocol_sha256']!=JUNE_PROTOCOL_SHA or digest_json(protocol_receipt['protocol'])!=JUNE_PROTOCOL_SHA:
        raise RuntimeError('The committed scientific protocol must remain unchanged.')
    if not re.fullmatch('[a-f0-9]{64}',recovery_sha):raise ValueError('Expected recovery source SHA256.')
    expected_revision='synthetic_fixture_only' if synthetic else SOURCE_REVISION
    root=Path(output_root or ('work/research/results/synthetic_june_recovery' if synthetic else 'work/research/results/june_recovery'))
    run=root/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ');run.mkdir(parents=True,exist_ok=False)
    disclosure={'version':'June-runtime-reset-rerun-v1','created_utc':datetime.now(timezone.utc).isoformat(),
        'reason':'The previous Colab runtime expired; its private predictions and source aggregate were lost.',
        'previous_June_source_inspection':{'duplicate_daily_key_groups':355,'extra_daily_rows':5680,
            'failed_groups_in_frozen_candidate_population':0,'ranking_performance_inspected':False},
        'original_implementation_sha256':ORIGINAL_JUNE_IMPLEMENTATION,
        'recovery_implementation_sha256':recovery_sha,'protocol_sha256':JUNE_PROTOCOL_SHA,
        'scientific_protocol_or_threshold_changes':False,'new_model_search_or_tuning':False,
        'source_scope':'All and only May-derived candidate keys, determined before outcome attachment.',
        'data_deduplication_performed':False,
        'interpretation':'Retrospective rerun after known June source-quality inspection; original prediction bytes/timestamps cannot be recovered or verified.'}
    write_json(run/'execution_amendment.json',disclosure);write_json(run/'protocol.json',protocol_receipt)
    progress('RUNTIME-RESET RERUN disclosed before execution. The original private checkpoints are unavailable.')
    candidate=Path(history_cache) if history_cache else Path('work/private_cache/study/development')/extractor_sha[:16]/'windows.parquet'
    historical=verified_cache(candidate,extractor_sha,synthetic)
    private=Path(private_root or ('work/private_cache/study/june_reset_synthetic' if synthetic else 'work/private_cache/study/june_reset_rerun'))/recovery_sha[:16]
    identity={'protocol_sha256':JUNE_PROTOCOL_SHA,'implementation_sha256':ORIGINAL_JUNE_IMPLEMENTATION,
        'historical_cache_sha256':historical['sha256'],'dataset_revision':expected_revision,
        'execution_recovery_sha256':recovery_sha,'runtime_reset_rerun':True}
    con=duckdb.connect()
    try:
        settings=memory_settings05m(con,private)
        validate_aggregate05m(con,candidate,merged=True)
        episodes={}
        for origin in JUNE_TRAINING:
            aggregate=con.execute('SELECT * FROM '+parquet05m([candidate])+' WHERE origin=?',[str(origin)]).df()
            episodes[origin],_=build_episode05(aggregate,origin)
        may=con.execute('SELECT * FROM '+parquet05m([candidate])+" WHERE origin='2026-05-01' AND role='outcome'").df()
        feature=june_feature_from_may(may);asof,_=build_episode05(feature,JUNE_ORIGIN)
        prediction_path=private/'predictions.parquet'
        reused_predictions=checked_checkpoint05m(prediction_path,identity) is not None
        records,frozen,prediction_receipt=freeze_predictions_june(con,episodes,asof,prediction_path,identity,progress)
        write_json(run/'predictions_frozen_receipt.json',prediction_receipt)
        raw_path=private/'global_june_aggregate.parquet'
        raw_identity={**identity,'kind':'global_June_source_preserved_before_candidate_quality_validation',
            'source_month':'2026-06','start_date':'2026-06-01','end_date':'2026-06-28'}
        raw_receipt=checked_checkpoint05m(raw_path,raw_identity)
        queried=raw_receipt is None
        if queried:
            if synthetic:
                if synthetic_source is None:raise ValueError('Local synthetic daily source required.')
                sources=[str(synthetic_source)];revision=lambda:expected_revision
            else:
                if synthetic_source is not None:raise ValueError('No source substitution in warehouse mode.')
                token=os.environ.get('HF_TOKEN')
                if not token:
                    from google.colab import userdata
                    token=userdata.get('HF_TOKEN')
                if not token or not re.fullmatch(r'hf_[A-Za-z0-9]+',token):raise RuntimeError('Enable the existing HF_TOKEN Colab Secret.')
                from huggingface_hub import HfApi
                api=HfApi(token=token)
                try:con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '"+token+"')")
                except Exception:raise RuntimeError('Could not configure warehouse access.') from None
                del token
                revision=lambda:api.dataset_info('FlyRank/internship-warehouse').sha
                sources=june_source_paths()
            if revision()!=expected_revision:raise RuntimeError('Source release differs from the fixed historical release.')
            started=datetime.now(timezone.utc).isoformat()
            con.register('_fix05_windows',june_windows())
            progress('Fixed rankings saved. Aggregating June 1-28 once; preserving the global quality audit.')
            raw_receipt=write_checkpoint05m(con,month_sql05m(sources,'2026-06'),raw_path,
                {**raw_identity,'acquisition_started_utc':started},validate=False)
            if revision()!=expected_revision:raise RuntimeError('Source release changed during acquisition; stop.')
            raw_receipt['source_revision_verified_after_query']=True
            write_json(raw_path.with_suffix('.json'),raw_receipt)
        if raw_receipt.get('source_revision_verified_after_query') is not True:
            raise RuntimeError('Global source checkpoint lacks a completed source-revision verification.')
        if datetime.fromisoformat(prediction_receipt['frozen_utc'])>datetime.fromisoformat(raw_receipt['acquisition_started_utc']):
            raise RuntimeError('Rerun predictions postdate the local outcome acquisition.')
        outcome_receipt=scope_cached_june_outcomes(con,raw_path,prediction_path,asof,
            private/'candidate_outcomes.parquet',identity)
        outcome=con.execute('SELECT * FROM '+parquet05m([private/'candidate_outcomes.parquet'])).df()
    finally:con.close()
    validation,paired,noise=report_june(episodes,feature,outcome,records,frozen,protocol_receipt['protocol'])
    execution={'mode':'synthetic_fixture_only' if synthetic else 'user_colab_warehouse',
        'implementation_sha256':ORIGINAL_JUNE_IMPLEMENTATION,'execution_recovery_sha256':recovery_sha,
        'historical_source':historical,'june_source':outcome_receipt,'global_june_source':raw_receipt,
        'june_partition_queried_this_run':bool(queried and not synthetic),
        'model_fits_this_run':0 if reused_predictions else sum(r['status']=='scored' for r in records),
        'original_prediction_checkpoint_recovered':False,'runtime_reset_rerun':True,
        'previous_June_source_inspection_disclosed':True,'independent_prospective_test':False,
        'prediction_frozen_utc':prediction_receipt['frozen_utc'],
        'this_rerun_outcome_acquisition_started_utc':raw_receipt['acquisition_started_utc'],
        'access_order_scope':'Only this rerun; June source defects were already known from the expired runtime.',
        'final_month_sample_used':False,'versions':PINS,'memory_settings':settings,
        'private_identifiers_or_prediction_arrays_exported':False,
        'all_candidate_source_quality_guards_passed':True}
    validation['execution']=execution
    for name,value in [('june_validation.json',validation),('paired_queue_audit.json',paired),('count_noise_audit.json',noise)]:
        write_json(run/name,value)
    write_json(run/'run_manifest.json',{'status':'complete_retrospective_rerun_with_disclosed_source_scoping',
        'execution':execution,'files':{p.name:file_hash05m(p) for p in sorted(run.glob('*.json'))},
        'scored_arms':sum(r['status']=='scored' for r in records)})
    progress('JUNE RERUN COMPLETE. Candidate quality checks passed; all fixed candidates retained.')
    return run


In [ ]:
EXTRACTION_SHA256 = 'f7dca08b36ac637ca8ea9c8c56f0184ccd234e8d6e3699311ae58cc4199f8974'
JUNE_RECOVERY_SHA256 = 'ca981a0fb289ca996c417ce7ee14568e5b2a19a412ccde07fea18cfb6b9fa3ef'
SYNTHETIC_JUNE = os.environ.get('FLYRANK_JUNE_SYNTHETIC') == '1'
june_run_directory = rerun_june_after_runtime_reset(JUNE_PROTOCOL_RECEIPT,
    EXTRACTION_SHA256,JUNE_RECOVERY_SHA256,synthetic=SYNTHETIC_JUNE,
    history_cache=os.environ.get('FLYRANK_JUNE_HISTORY_CACHE'),
    synthetic_source=os.environ.get('FLYRANK_JUNE_SYNTHETIC_SOURCE'))


In [ ]:
june_figure_path = plot_june_results(june_run_directory)
from IPython.display import display, Image
display(Image(filename=str(june_figure_path)))
print((june_run_directory/'figure_caption.txt').read_text(encoding='utf-8'))
_june_result = json.loads((june_run_directory/'june_validation.json').read_text(encoding='utf-8'))
_june_table=[]
for _arm in _june_result['folds'][0]['arms']:
    print(_arm['arm'],_arm['status'])
    for _row in _arm['metrics']:
        if _row['minimum_prior_clicks']!=1:continue
        _queue=next(q for q in _row['queues'] if q['budget']==50)
        _june_table.append({'arm':_arm['arm'],'method':_row['method'],
            'observed_hits':_queue.get('observed_hits'),'unknown':_queue.get('unknown_outcomes'),
            'P50_lower':_queue.get('precision_lower_bound'),'P50_upper':_queue.get('precision_upper_bound')})
display(pd.DataFrame(_june_table))


In [ ]:
# Export only aggregate results and the existing comparison figure.
import zipfile
_names=['protocol.json','execution_amendment.json','predictions_frozen_receipt.json',
    'june_validation.json','paired_queue_audit.json','count_noise_audit.json',
    'run_manifest.json','june_summary.png','june_summary.svg','figure_caption.txt']
_present=[june_run_directory/n for n in _names if (june_run_directory/n).is_file()]
write_json(june_run_directory/'artifact_checksums.json',{p.name:file_hash05m(p) for p in _present})
june_archive=Path('FlyRank_June_recovery_results.zip')
with zipfile.ZipFile(june_archive,'w',zipfile.ZIP_DEFLATED) as _bundle:
    for _path in _present+[june_run_directory/'artifact_checksums.json']:_bundle.write(_path,_path.name)
print('JUNE RECOVERY NOTEBOOK COMPLETE. Retrospective rerun; previous runtime checkpoints were lost.')
if not SYNTHETIC_JUNE:
    from google.colab import files
    files.download(str(june_archive))
